# 04 · Train on the GPU: a device-resident loop, checkpoints, evaluation and the capstone

*Base Model Training in C, C++ and CUDA*, notebook 4 of 4

In notebook 3 the GPU computed one forward pass and handed every result back to the host. Training repeats a forward pass, a backward pass and an update thousands of times, so in this notebook nothing leaves the device unless it has to: the training set, the weights, the gradients, Adam's state and every intermediate result stay in GPU memory, and the row numbers of every mini-batch go in with one copy before the loop. You write the remaining kernels and the backward half of a training step, verify the whole loop against an independent double-precision implementation, measure where the GPU wins, save checkpoints that resume exactly, and finish with a capstone: a small, reproducible experiment with one controlled change, a single test evaluation and a verified artefact.

## What you will build

- `relu_backward_kernel` and `adam_kernel`: the last two kernels that training needs;
- `backward_and_update`: the launch sequence of the backward pass and the optimiser step, on the device;
- `confusion_matrix`: errors class by class, for the final evaluation;
- the capstone: a protocol written in advance, a comparison across seeds on validation data, one test evaluation and a reload check.

## Learning objectives

By the end of this notebook you will be able to:

1. design a training loop whose data, weights, gradients and optimiser state stay in device memory, and estimate its memory;
2. write the backward pass and an Adam update as kernel launches, and check the result against an independent implementation;
3. explain when a GPU is faster than a CPU, and how transfers and synchronisation slow a training loop;
4. save GPU training state to a checkpoint and resume it exactly;
5. select a checkpoint on validation data, test once, and verify a reloaded artefact;
6. run, report and defend a small reproducible experiment, stating its backend and its limits.

### How to work through this notebook

Run the cells from top to bottom. At each **✍️ Your turn** you write code inside the string that a small Python function returns (or, once, a Python function): delete the line `raise NotImplementedError`, type your code where the comment says, and run the check cell beneath it.

- ✅ means your code matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not compile or does not match yet. In both cases the notebook carries on with a hidden reference version, so *Run All* always works and you can come back later.

Each check compiles your code into a small test program and runs it in its own process. Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 0 for Python:** the standard library only. NumPy is not used: every number of the model is computed by your kernels.
- **An NVIDIA GPU with the CUDA toolkit** runs the kernels for real. In **Google Colab**, choose *Runtime → Change runtime type → T4 GPU* (or any GPU) before running the setup cells.
- **Without a GPU**, the CPU emulator from notebook 3 runs the same CUDA source on the processor, so every check and result means the same thing; only the timings become CPU timings, and the cells say so. The course notebook server runs it this way.
- **Not in the browser preview:** JupyterLite cannot run a compiler.
- Checkpoints go to a temporary folder that is removed when the kernel stops.
- You should have finished notebook 3.

## Setup

Run these two cells first: the tools from the earlier notebooks, then the choice of CUDA backend (a GPU with `nvcc`, or the CPU emulator).

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import atexit
import ctypes
import hashlib
import math
import os
import random
import shutil
import signal
import subprocess
import sys
import tempfile
from array import array

if sys.platform == "emscripten":
    raise RuntimeError(
        "This notebook compiles C, C++ and CUDA, which a web browser cannot do. Open it on your "
        "course notebook server, in Google Colab, or in Jupyter on a computer with a C compiler."
    )

CC = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
CXX = shutil.which("c++") or shutil.which("g++") or shutil.which("clang++")
if not (CC and CXX):
    raise RuntimeError("No C and C++ compiler found: install gcc and g++ (or clang), then restart the kernel.")
BUILD_DIR = tempfile.mkdtemp(prefix="furina-native-")
atexit.register(shutil.rmtree, BUILD_DIR, True)
COMPILED = {}  # source hash -> path of the program or library built from it


class CompileError(Exception):
    """The compiler rejected the code. The message is what the compiler printed."""


class RunError(Exception):
    """A compiled program crashed, never finished, or stopped with an error."""


def _trim(text, lines=14):
    kept = text.strip().splitlines()
    return "\n".join(kept[:lines] + (["   ..."] if len(kept) > lines else []))


def mark(source, label):
    """Splice in source so that compiler messages name `label` and count from its first line."""
    return f'\n#line 1 "{label}"\n{source}\n'


def build(source, language="c", shared=False, flags=()):
    """Compile C ("c"), C++ ("c++") or CUDA ("cuda") into a program, or a shared library."""
    key = hashlib.sha256(f"{language}|{shared}|{flags}|{source}".encode()).hexdigest()[:24]
    if key in COMPILED:
        return COMPILED[key]
    output = os.path.join(BUILD_DIR, key + (".so" if shared else ".exe"))
    if language == "cuda":
        command = cuda_command(source, key, output)  # defined in the CUDA notebooks
    else:
        path = os.path.join(BUILD_DIR, key + (".c" if language == "c" else ".cpp"))
        with open(path, "w", encoding="utf-8") as handle:
            handle.write(source)
        if language == "c":
            command = [CC, "-std=c11", "-O2", "-ffp-contract=off", "-Wall", path, "-o", output, "-lm"]
        else:
            command = [CXX, "-std=c++17", "-O2", "-ffp-contract=off", "-Wall", path, "-o", output]
    if shared:
        command[1:1] = ["-shared"] + (["-Xcompiler", "-fPIC"] if command[0] == globals().get("NVCC") else ["-fPIC"])
    command += list(flags)  # last, so that a flag such as -O1 overrides the default
    result = subprocess.run(command, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        raise CompileError(_trim(result.stderr or result.stdout))
    COMPILED[key] = output
    return output


def load(source, language="c"):
    """Compile source into a shared library and load it into this Python process."""
    return ctypes.CDLL(build(source, language, shared=True))


def run(program, timeout=20, env=None):
    """Run a compiled program in its own process and return what it printed."""
    try:
        result = subprocess.run([program], capture_output=True, text=True, timeout=timeout,
                                env=None if env is None else {**os.environ, **env})
    except subprocess.TimeoutExpired:
        raise RunError(f"did not finish within {timeout} seconds: is there a loop that never ends?") from None
    if result.returncode < 0:
        try:
            name = signal.Signals(-result.returncode).name
        except ValueError:
            name = f"signal {-result.returncode}"
        hint = {
            "SIGSEGV": "a read or write outside memory the program owns, such as an index past the end of an array",
            "SIGBUS": "a read or write outside memory the program owns",
            "SIGFPE": "an integer division by zero",
            "SIGABRT": "the program aborted itself, often after finding corrupted memory",
            "SIGTRAP": "the compiler turned undefined behaviour, such as using a null pointer, into a trap",
            "SIGILL": "the compiler turned undefined behaviour, such as using a null pointer, into a trap",
        }.get(name, "")
        detail = f"\n{_trim(result.stderr)}" if result.stderr.strip() else ""
        raise RunError(f"crashed with {name}" + (f" ({hint})" if hint else "") + detail)
    if result.returncode != 0:
        raise RunError(f"stopped with exit status {result.returncode}:\n{_trim(result.stderr + result.stdout)}")
    return result.stdout


def _value(token):
    try:
        return float(token)
    except ValueError:
        return None


def _same(a, b, atol, rtol):
    if a == b:
        return True
    x, y = _value(a), _value(b)
    if x is None or y is None:
        return False
    if math.isnan(x) or math.isnan(y):
        return math.isnan(x) and math.isnan(y)
    return math.isclose(x, y, rel_tol=rtol, abs_tol=atol)


def _short(text, limit=110):
    text = " ".join(text.split())
    return text if len(text) <= limit else text[: limit - 3] + "..."


def compare_output(got, want, atol=1e-9, rtol=1e-9):
    """None when two printouts match line by line, otherwise the first difference."""
    got_lines, want_lines = got.strip().splitlines(), want.strip().splitlines()
    for got_line, want_line in zip(got_lines, want_lines):
        got_label, _, got_values = got_line.partition(":")
        want_label, _, want_values = want_line.partition(":")
        g, w = got_values.split(), want_values.split()
        if got_label != want_label or len(g) != len(w) or not all(_same(a, b, atol, rtol) for a, b in zip(g, w)):
            return (f"for {want_label.strip() or 'a case'}\n   your code printed {_short(got_values) or '(nothing)'}"
                    f"\n   expected          {_short(want_values)}")
    if len(got_lines) < len(want_lines):
        return f"your program stopped before printing {want_lines[len(got_lines)].partition(':')[0]!r}"
    if len(got_lines) > len(want_lines):
        return f"your program printed {len(got_lines) - len(want_lines)} line(s) more than expected"
    return None


REFERENCE_OUTPUT = {}


def practice_native(name, yours, reference, program, language="c", atol=1e-9, rtol=1e-9, timeout=20):
    """Check native code you wrote against the reference.

    `yours` and `reference` return source code. `program(source)` wraps a
    source in a complete program whose main() runs the cases and prints the
    results. Both programs are compiled and run in their own processes, so a
    crash or an endless loop fails the check without stopping this notebook.
    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference, so you can always keep going.
    """
    fallback = "   The rest of the notebook uses the reference version for now."
    key = hashlib.sha256(f"{language}|{program(reference())}".encode()).hexdigest()
    if key not in REFERENCE_OUTPUT:
        REFERENCE_OUTPUT[key] = run(build(program(reference()), language), timeout)
    try:
        source = yours()
        if not isinstance(source, str):
            raise TypeError(f"{name} must return its source code as a string, not {type(source).__name__}")
        got = run(build(program(source), language), timeout)
    except NotImplementedError:
        print(f"✍️ {name}: not written yet.")
        print(fallback)
        return reference
    except CompileError as error:
        print(f"❌ {name}: the compiler rejected your code:\n{error}")
        print(fallback)
        return reference
    except RunError as error:
        print(f"❌ {name}: the check program {error}")
        print(fallback)
        return reference
    except Exception as error:  # a mistake in the Python around the source
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(fallback)
        return reference
    difference = compare_output(got, REFERENCE_OUTPUT[key], atol, rtol)
    if difference:
        print(f"❌ {name}: {difference}")
        print(fallback)
        return reference
    print(f"✅ {name}: matches the reference. From here on, the notebook uses your version.")
    return yours


def _close(got, want, atol):
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want))
    if isinstance(want, float) or isinstance(got, float):
        try:
            return math.isclose(float(got), float(want), rel_tol=1e-9, abs_tol=atol)
        except (TypeError, ValueError):
            return False
    return got == want


def practice(name, yours, reference, cases, atol=1e-9):
    """Check a Python function you wrote against the reference on a few inputs."""
    fallback = "   The rest of the notebook uses the reference version for now."
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(repr(args))}")
                print(f"   you returned {_short(repr(got))}")
                print(f"   expected     {_short(repr(want))}")
                print(fallback)
                return reference
    except NotImplementedError:
        print(f"✍️ {name}: not written yet.")
        print(fallback)
        return reference
    except Exception as error:
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(fallback)
        return reference
    print(f"✅ {name}: matches the reference. From here on, the notebook uses your version.")
    return yours


def address(buffer):
    """The memory address of an array.array, to pass to C as a pointer."""
    return buffer.buffer_info()[0]


def c_list(values, kind="double"):
    """Python numbers as a C initialiser list, such as {1.5, -2.0}."""
    if kind == "int":
        return "{" + ", ".join(str(int(v)) for v in values) + "}"
    suffix = "f" if kind == "float" else ""
    return "{" + ", ".join(f"{float(v)!r}{suffix}" for v in values) + "}"


print(f"C compiler: {CC}\nC++ compiler: {CXX}")

In [ ]:
# ── The CUDA backend: a real GPU when there is one, the CPU emulator otherwise ──
import re

CUDA_CPU_HEADER = r'''
// furina_cuda_cpu.h: runs CUDA C++ on the CPU when no NVIDIA GPU is present.
//
// The course compiles every kernel with nvcc and runs it on the GPU whenever
// the machine has one. This header is the fallback for CPU-only machines: the
// same source then compiles with an ordinary C++ compiler, and every kernel
// runs on the CPU with CUDA's indexing rules, so results can be checked.
//
//   * __global__, __device__ and __host__ functions become ordinary functions.
//   * Before compiling, the notebook rewrites `kernel<<<grid, block>>>(args)`
//     into `furina_launch<SYNC>(kernel, grid, block)(args)`. The launch runs the
//     kernel once for every thread of every block, with threadIdx, blockIdx,
//     blockDim and gridDim set as on a GPU. Blocks run one after another.
//   * A kernel that calls __syncthreads() (SYNC is true) runs each block with
//     one operating-system thread per CUDA thread, so the barrier really
//     waits. Other kernels run their threads one at a time, which is faster.
//   * __shared__ variables become static: blocks run one at a time, so all
//     threads of the running block share one copy, as on a GPU.
//   * cudaMalloc returns host memory filled with 0xFF bytes, which reads as
//     NaN in float or double, so forgetting to initialise memory shows up.
//
// It does not model speed, run blocks in parallel, emulate warps (__shfl_*),
// dynamic shared memory or streams, or find data races. Timings measured
// with it are CPU timings.
#pragma once

#include <cmath>
#include <cstddef>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <pthread.h>
#include <time.h>

#define __global__
#define __device__
#define __host__
#define __forceinline__ inline
#define __launch_bounds__(...)
#define __restrict__ __restrict
#define __shared__ static

struct uint3 { unsigned int x, y, z; };
struct dim3 {
    unsigned int x, y, z;
    dim3(unsigned int x_ = 1, unsigned int y_ = 1, unsigned int z_ = 1) : x(x_), y(y_), z(z_) {}
};

static thread_local uint3 threadIdx = {0, 0, 0};
static thread_local uint3 blockIdx = {0, 0, 0};
static dim3 blockDim, gridDim;
static const int warpSize = 32;

// ------------------------------------------------------------------ errors

enum cudaError_t {
    cudaSuccess = 0,
    cudaErrorInvalidValue = 1,
    cudaErrorMemoryAllocation = 2,
    cudaErrorInvalidConfiguration = 9,
    cudaErrorInvalidMemcpyDirection = 21,
    cudaErrorLaunchOutOfResources = 701,
    cudaErrorLaunchFailure = 719,
};

static cudaError_t furina_error = cudaSuccess;

static inline cudaError_t furina_fail(cudaError_t error) {
    if (furina_error == cudaSuccess) furina_error = error;
    return error;
}

static inline cudaError_t cudaGetLastError() {
    cudaError_t error = furina_error;
    furina_error = cudaSuccess;
    return error;
}

static inline cudaError_t cudaPeekAtLastError() { return furina_error; }

static inline const char* cudaGetErrorString(cudaError_t error) {
    switch (error) {
        case cudaSuccess: return "no error";
        case cudaErrorInvalidValue: return "invalid argument";
        case cudaErrorMemoryAllocation: return "out of memory";
        case cudaErrorInvalidConfiguration: return "invalid configuration argument";
        case cudaErrorInvalidMemcpyDirection: return "invalid copy direction for memcpy";
        case cudaErrorLaunchOutOfResources: return "too many resources requested for launch";
        case cudaErrorLaunchFailure: return "unspecified launch failure";
    }
    return "unknown error";
}

static inline cudaError_t cudaDeviceSynchronize() { return furina_error; }
static inline cudaError_t cudaSetDevice(int) { return cudaSuccess; }
static inline cudaError_t cudaGetDeviceCount(int* count) { *count = 1; return cudaSuccess; }

struct cudaDeviceProp {
    char name[256];
    size_t totalGlobalMem, sharedMemPerBlock;
    int warpSize, maxThreadsPerBlock, multiProcessorCount, major, minor;
    int maxThreadsDim[3], maxGridSize[3];
};

static inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* prop, int) {
    std::memset(prop, 0, sizeof *prop);
    std::snprintf(prop->name, sizeof prop->name, "CPU emulator (no NVIDIA GPU)");
    prop->totalGlobalMem = (size_t)1 << 30;
    prop->sharedMemPerBlock = 48 * 1024;
    prop->warpSize = 32;
    prop->maxThreadsPerBlock = 1024;
    prop->multiProcessorCount = 1;
    prop->maxThreadsDim[0] = 1024; prop->maxThreadsDim[1] = 1024; prop->maxThreadsDim[2] = 64;
    prop->maxGridSize[0] = 2147483647; prop->maxGridSize[1] = 65535; prop->maxGridSize[2] = 65535;
    return cudaSuccess;
}

// ------------------------------------------------------------------ memory

enum cudaMemcpyKind {
    cudaMemcpyHostToHost = 0,
    cudaMemcpyHostToDevice = 1,
    cudaMemcpyDeviceToHost = 2,
    cudaMemcpyDeviceToDevice = 3,
    cudaMemcpyDefault = 4,
};

// Live "device" allocations, so that cudaMemcpy can reject a copy whose
// direction does not match its pointers, as a GPU would.
struct FurinaAllocation { const char* begin; size_t bytes; };
static FurinaAllocation furina_allocations[4096];
static int furina_allocation_count = 0;

static inline bool furina_is_device(const void* pointer) {
    const char* p = static_cast<const char*>(pointer);
    for (int i = 0; i < furina_allocation_count; ++i) {
        const FurinaAllocation& a = furina_allocations[i];
        if (p >= a.begin && p < a.begin + (a.bytes ? a.bytes : 1)) return true;
    }
    return false;
}

static inline cudaError_t cudaMalloc(void** pointer, size_t bytes) {
    if (furina_allocation_count == 4096) return furina_fail(cudaErrorMemoryAllocation);
    void* memory = std::malloc(bytes ? bytes : 1);
    if (!memory) return furina_fail(cudaErrorMemoryAllocation);
    std::memset(memory, 0xFF, bytes);
    furina_allocations[furina_allocation_count++] = {static_cast<const char*>(memory), bytes};
    *pointer = memory;
    return cudaSuccess;
}

template <class T>
static inline cudaError_t cudaMalloc(T** pointer, size_t bytes) {
    return cudaMalloc(reinterpret_cast<void**>(pointer), bytes);
}

static inline cudaError_t cudaFree(void* pointer) {
    if (!pointer) return cudaSuccess;
    for (int i = 0; i < furina_allocation_count; ++i) {
        if (furina_allocations[i].begin == pointer) {
            furina_allocations[i] = furina_allocations[--furina_allocation_count];
            std::free(pointer);
            return cudaSuccess;
        }
    }
    return furina_fail(cudaErrorInvalidValue);
}

static inline cudaError_t cudaMemcpy(void* dst, const void* src, size_t bytes, cudaMemcpyKind kind) {
    bool dst_device = furina_is_device(dst), src_device = furina_is_device(src);
    bool ok = kind == cudaMemcpyDefault
        || (kind == cudaMemcpyHostToDevice && dst_device && !src_device)
        || (kind == cudaMemcpyDeviceToHost && !dst_device && src_device)
        || (kind == cudaMemcpyDeviceToDevice && dst_device && src_device)
        || (kind == cudaMemcpyHostToHost && !dst_device && !src_device);
    if (!ok) return furina_fail(cudaErrorInvalidMemcpyDirection);
    if (bytes) std::memmove(dst, src, bytes);
    return cudaSuccess;
}

static inline cudaError_t cudaMemset(void* pointer, int value, size_t bytes) {
    if (!furina_is_device(pointer)) return furina_fail(cudaErrorInvalidValue);
    std::memset(pointer, value, bytes);
    return cudaSuccess;
}

// ------------------------------------------------------------------ timing

struct FurinaEvent { double milliseconds; };
typedef FurinaEvent* cudaEvent_t;

static inline double furina_now_ms() {
    struct timespec now;
    clock_gettime(CLOCK_MONOTONIC, &now);
    return now.tv_sec * 1e3 + now.tv_nsec / 1e6;
}

static inline cudaError_t cudaEventCreate(cudaEvent_t* event) { *event = new FurinaEvent{0.0}; return cudaSuccess; }
static inline cudaError_t cudaEventRecord(cudaEvent_t event, int stream = 0) { (void)stream; event->milliseconds = furina_now_ms(); return cudaSuccess; }
static inline cudaError_t cudaEventSynchronize(cudaEvent_t) { return cudaSuccess; }
static inline cudaError_t cudaEventElapsedTime(float* ms, cudaEvent_t start, cudaEvent_t end) {
    *ms = static_cast<float>(end->milliseconds - start->milliseconds);
    return cudaSuccess;
}
static inline cudaError_t cudaEventDestroy(cudaEvent_t event) { delete event; return cudaSuccess; }

// ------------------------------------------------- device-side functions

static inline float atomicAdd(float* address, float value) {
    uint32_t* word = reinterpret_cast<uint32_t*>(address);
    uint32_t seen = __atomic_load_n(word, __ATOMIC_RELAXED), wanted;
    float before, after;
    do {
        std::memcpy(&before, &seen, sizeof before);
        after = before + value;
        std::memcpy(&wanted, &after, sizeof wanted);
    } while (!__atomic_compare_exchange_n(word, &seen, wanted, false, __ATOMIC_SEQ_CST, __ATOMIC_RELAXED));
    return before;
}

static inline double atomicAdd(double* address, double value) {
    uint64_t* word = reinterpret_cast<uint64_t*>(address);
    uint64_t seen = __atomic_load_n(word, __ATOMIC_RELAXED), wanted;
    double before, after;
    do {
        std::memcpy(&before, &seen, sizeof before);
        after = before + value;
        std::memcpy(&wanted, &after, sizeof wanted);
    } while (!__atomic_compare_exchange_n(word, &seen, wanted, false, __ATOMIC_SEQ_CST, __ATOMIC_RELAXED));
    return before;
}

static inline int atomicAdd(int* address, int value) { return __atomic_fetch_add(address, value, __ATOMIC_SEQ_CST); }

static inline int min(int a, int b) { return a < b ? a : b; }
static inline int max(int a, int b) { return a > b ? a : b; }
static inline float min(float a, float b) { return a < b ? a : b; }
static inline float max(float a, float b) { return a > b ? a : b; }
static inline double min(double a, double b) { return a < b ? a : b; }
static inline double max(double a, double b) { return a > b ? a : b; }
inline float __expf(float x) { return expf(x); }
inline float __logf(float x) { return logf(x); }
inline float rsqrtf(float x) { return 1.0f / sqrtf(x); }

// ---------------------------------------------------------------- barriers

struct FurinaBlock {
    pthread_mutex_t mutex;
    pthread_cond_t changed;
    unsigned int live;          // threads of this block that have not returned
    unsigned int arrived;       // threads waiting at the current barrier
    unsigned long generation;   // how many times the barrier has opened
};

static FurinaBlock* furina_block = nullptr;  // the block running on threads, if any
struct FurinaUnexpectedSync {};

static inline void furina_open(FurinaBlock* block) {
    block->arrived = 0;
    block->generation++;
    pthread_cond_broadcast(&block->changed);
}

static inline void __syncthreads() {
    FurinaBlock* block = furina_block;
    if (!block) throw FurinaUnexpectedSync{};
    pthread_mutex_lock(&block->mutex);
    unsigned long generation = block->generation;
    if (++block->arrived >= block->live) {
        furina_open(block);
    } else {
        while (generation == block->generation) pthread_cond_wait(&block->changed, &block->mutex);
    }
    pthread_mutex_unlock(&block->mutex);
}

// A thread that has returned counts as arrived at every later barrier, so a
// block whose threads leave early does not wait forever.
static inline void furina_thread_finished(FurinaBlock* block) {
    pthread_mutex_lock(&block->mutex);
    block->live--;
    if (block->arrived > 0 && block->arrived >= block->live) furina_open(block);
    pthread_mutex_unlock(&block->mutex);
}

template <class Body>
struct FurinaThread {
    Body* body;
    FurinaBlock* block;
    uint3 thread, index;
};

template <class Body>
static void* furina_thread_main(void* raw) {
    FurinaThread<Body>* start = static_cast<FurinaThread<Body>*>(raw);
    threadIdx = start->thread;
    blockIdx = start->index;
    (*start->body)();
    furina_thread_finished(start->block);
    return nullptr;
}

// Runs one block with one OS thread per CUDA thread; false if the threads
// could not all be started (the notebook server limits how many may run).
template <class Body>
static bool furina_run_block_on_threads(Body& body, uint3 index, dim3 block) {
    unsigned int count = block.x * block.y * block.z;
    FurinaBlock state;
    pthread_mutex_init(&state.mutex, nullptr);
    pthread_cond_init(&state.changed, nullptr);
    state.live = count;
    state.arrived = 0;
    state.generation = 0;
    FurinaThread<Body>* starts = new FurinaThread<Body>[count];
    pthread_t* threads = new pthread_t[count];
    pthread_attr_t attributes;
    pthread_attr_init(&attributes);
    pthread_attr_setstacksize(&attributes, 256 * 1024);
    furina_block = &state;
    unsigned int started = 0;
    for (; started < count; ++started) {
        uint3 thread = {started % block.x, (started / block.x) % block.y, started / (block.x * block.y)};
        starts[started] = {&body, &state, thread, index};
        if (pthread_create(&threads[started], &attributes, furina_thread_main<Body>, &starts[started]) != 0) break;
    }
    if (started < count) {
        // Threads already running may be waiting for the ones that never
        // started: count those as finished so that the block can drain.
        pthread_mutex_lock(&state.mutex);
        state.live -= count - started;
        if (state.arrived > 0 && state.arrived >= state.live) furina_open(&state);
        pthread_mutex_unlock(&state.mutex);
        std::fprintf(stderr,
            "CPU emulator: could only start %u of the %u threads of a block that calls __syncthreads(); "
            "use a smaller block, such as 8 x 8.\n", started, count);
    }
    for (unsigned int i = 0; i < started; ++i) pthread_join(threads[i], nullptr);
    furina_block = nullptr;
    pthread_attr_destroy(&attributes);
    pthread_cond_destroy(&state.changed);
    pthread_mutex_destroy(&state.mutex);
    delete[] threads;
    delete[] starts;
    return started == count;
}

// ------------------------------------------------------------------ launch

template <bool Sync, class Body>
static void furina_run(dim3 grid, dim3 block, size_t shared, Body body) {
    unsigned long long threads = 1ULL * block.x * block.y * block.z;
    if (shared != 0) {
        std::fprintf(stderr, "CPU emulator: dynamic shared memory (a third launch argument) is not supported.\n");
        furina_fail(cudaErrorInvalidValue);
        return;
    }
    if (threads == 0 || threads > 1024 || block.z > 64 || grid.x == 0 || grid.y == 0 || grid.z == 0
        || grid.y > 65535 || grid.z > 65535) {
        furina_fail(cudaErrorInvalidConfiguration);
        return;
    }
    gridDim = grid;
    blockDim = block;
    for (unsigned int bz = 0; bz < grid.z; ++bz)
        for (unsigned int by = 0; by < grid.y; ++by)
            for (unsigned int bx = 0; bx < grid.x; ++bx) {
                uint3 index = {bx, by, bz};
                if (Sync) {
                    if (!furina_run_block_on_threads(body, index, block)) {
                        furina_fail(cudaErrorLaunchOutOfResources);
                        return;
                    }
                    continue;
                }
                blockIdx = index;
                try {
                    for (unsigned int tz = 0; tz < block.z; ++tz)
                        for (unsigned int ty = 0; ty < block.y; ++ty)
                            for (unsigned int tx = 0; tx < block.x; ++tx) {
                                threadIdx = {tx, ty, tz};
                                body();
                            }
                } catch (FurinaUnexpectedSync&) {
                    std::fprintf(stderr, "CPU emulator: __syncthreads() was called outside a kernel that "
                                         "contains it; call it directly in the kernel's body.\n");
                    furina_fail(cudaErrorLaunchFailure);
                    return;
                }
            }
}

template <bool Sync, class... Params>
struct FurinaLaunch {
    void (*kernel)(Params...);
    dim3 grid, block;
    size_t shared;
    template <class... Args>
    void operator()(Args... args) const {
        void (*k)(Params...) = kernel;
        furina_run<Sync>(grid, block, shared, [=]() { k(args...); });
    }
};

template <bool Sync, class... Params>
static FurinaLaunch<Sync, Params...> furina_launch(void (*kernel)(Params...), dim3 grid, dim3 block,
                                                   size_t shared = 0, int stream = 0) {
    (void)stream;
    return {kernel, grid, block, shared};
}
'''

_COMMENT = re.compile(r"//[^\n]*|/\*.*?\*/|\"(?:\\.|[^\"\\\n])*\"|'(?:\\.|[^'\\\n])*'", re.S)
_DEFINITION = re.compile(r"__(?:global|device)__[^;{(]*?\b([A-Za-z_]\w*)\s*\(")
_LAUNCH = re.compile(r"\b([A-Za-z_]\w*)\s*<<<(.*?)>>>\s*\(", re.S)


def kernels_that_sync(source):
    # Kernels and __device__ functions that call __syncthreads(), directly or through a helper.
    code = _COMMENT.sub(lambda m: m.group(0) if m.group(0)[0] in "\"'" else " " * len(m.group(0)), source)
    bodies = {}
    for match in _DEFINITION.finditer(code):
        depth, i = 1, match.end()
        while i < len(code) and depth:
            depth += {"(": 1, ")": -1}.get(code[i], 0)
            i += 1
        while i < len(code) and code[i].isspace():
            i += 1
        if i < len(code) and code[i] == "{":
            start, depth = i, 0
            while i < len(code):
                depth += {"{": 1, "}": -1}.get(code[i], 0)
                i += 1
                if depth == 0:
                    break
            bodies[match.group(1)] = code[start:i]
    syncing = {name for name, body in bodies.items() if "__syncthreads" in body}
    grew = True
    while grew:
        grew = False
        for name, body in bodies.items():
            if name not in syncing and any(re.search(rf"\b{helper}\s*\(", body) for helper in syncing):
                syncing.add(name)
                grew = True
    return syncing


def emulate_launches(source):
    # Rewrite kernel<<<grid, block>>>(args) as furina_launch<sync>(kernel, grid, block)(args).
    syncing = kernels_that_sync(source)
    return _LAUNCH.sub(
        lambda m: f"furina_launch<{'true' if m.group(1) in syncing else 'false'}>({m.group(1)}, {m.group(2)})(",
        source)


def _find_gpu():
    nvcc = shutil.which("nvcc") or next(
        (path for path in ("/usr/local/cuda/bin/nvcc",) if os.path.exists(path)), None)
    smi = shutil.which("nvidia-smi")
    if not (nvcc and smi):
        return None, None
    try:
        listing = subprocess.run([smi, "-L"], capture_output=True, text=True, timeout=30).stdout
    except (OSError, subprocess.SubprocessError):
        return None, None
    gpus = [line for line in listing.splitlines() if line.startswith("GPU ")]
    return (nvcc, gpus[0].split(" (UUID")[0]) if gpus else (None, None)


def cuda_command(source, key, output):
    path = os.path.join(BUILD_DIR, key + (".cu" if CUDA_BACKEND == "gpu" else ".cpp"))
    with open(path, "w", encoding="utf-8") as handle:
        handle.write(source if CUDA_BACKEND == "gpu" else emulate_launches(source))
    if CUDA_BACKEND == "gpu":
        return [NVCC, "-O2", f"-arch={NVCC_ARCH}", path, "-o", output]
    return [CXX, "-std=c++17", "-O2", "-ffp-contract=off", "-pthread", "-include", HEADER_PATH, path, "-o", output]


HEADER_PATH = os.path.join(BUILD_DIR, "furina_cuda_cpu.h")
with open(HEADER_PATH, "w", encoding="utf-8") as handle:
    handle.write(CUDA_CPU_HEADER)

_choice = os.environ.get("FURINA_CUDA_BACKEND", "auto").strip().lower()
NVCC, GPU_NAME = (None, None) if _choice == "emulator" else _find_gpu()
if _choice == "gpu" and not NVCC:
    raise RuntimeError("FURINA_CUDA_BACKEND=gpu, but no NVIDIA GPU with nvcc was found.")
CUDA_BACKEND = "gpu" if NVCC else "emulator"
NVCC_ARCH = "native"
if CUDA_BACKEND == "gpu":
    probe = os.path.join(BUILD_DIR, "probe.cu")
    with open(probe, "w", encoding="utf-8") as handle:
        handle.write("__global__ void k() {}\nint main() { k<<<1, 1>>>(); return cudaDeviceSynchronize(); }\n")
    if subprocess.run([NVCC, "-arch=native", probe, "-o", probe + ".exe"], capture_output=True).returncode:
        NVCC_ARCH = "sm_60"  # an nvcc older than 11.6 has no -arch=native
    print(f"CUDA backend: NVIDIA GPU, {GPU_NAME}. Kernels compile with {NVCC} and run on the GPU.")
else:
    print("CUDA backend: CPU emulator (no NVIDIA GPU with nvcc was found). Kernels compile with the C++ "
          "compiler and run on the CPU with CUDA's indexing rules. Results are checked as on a GPU; "
          "timings are CPU timings, not GPU performance.")

## 📦 Recap: what you built earlier

The hidden cell below holds the kernels from notebook 3, in their reference versions: `relu_kernel`, `matmul_kernel`, `softmax_xent_kernel`, `column_sums_kernel`, `add_bias_kernel` and `sum_kernel`. It also adds three new kernels: the two **transposed** products the backward pass needs, $C = A^\top B$ and $C = A B^\top$, which read their operands in transposed order instead of building transposes, and `gather_rows_kernel`, which copies the rows of a mini-batch out of the training set on the device. Finally, it defines two preludes: `CHECK4` for check programs, which stop at the first CUDA error, and `LIB4` for the library loaded into Python, which records the error and reports it instead. Run it; open it only if you want to look back at the code.

In [ ]:
CHECK4 = r"""
#include <cmath>
#include <cstdio>
#include <cstdlib>

#ifndef TILE
#define TILE 8          // 8 x 8 = 64 threads per tiled block; GPUs usually use 16 or 32
#endif

// Stops the check program with a readable message when a CUDA call fails.
#define CUDA_CHECK(call)                                                                   \
    do {                                                                                   \
        cudaError_t error_ = (call);                                                       \
        if (error_ != cudaSuccess) {                                                       \
            std::printf("CUDA error: %s (line %d)\n", cudaGetErrorString(error_), __LINE__); \
            std::exit(1);                                                                  \
        }                                                                                  \
    } while (0)

static void show(const char* label, const float* x, int n)
{
    std::printf("%s:", label);
    for (int i = 0; i < n; ++i)
        std::printf(" %.9g", x[i]);
    std::printf("\n");
}

static void fill(float* x, int n, float seed)         // the same test numbers on every machine
{
    for (int i = 0; i < n; ++i)
        x[i] = sinf(seed + 0.37f * i);
}

static void clear_errors() {}                  // a check program stops at its first error
static int cuda_status() { return 0; }
"""

LIB4 = r"""
#include <cmath>
#include <cstdio>

#ifndef TILE
#define TILE 8
#endif

// Inside a library loaded into Python, a failing CUDA call must not end the
// process: CUDA_CHECK records the first error, and each host function reports it.
static cudaError_t first_error = cudaSuccess;
static void record_error(cudaError_t error)
{
    if (error != cudaSuccess && first_error == cudaSuccess)
        first_error = error;
}
#define CUDA_CHECK(call) record_error(call)

extern "C" const char* last_cuda_error() { return cudaGetErrorString(first_error); }

static void clear_errors()
{
    first_error = cudaSuccess;
    cudaGetLastError();
}
static int cuda_status() { return first_error == cudaSuccess ? 0 : 1; }
"""

RECAP_CU = r"""
__global__ void relu_kernel(const float* x, float* y, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        y[i] = x[i] > 0.0f ? x[i] : 0.0f;
}

void relu_gpu(const float* x, float* y, int n)
{
    if (n == 0)
        return;
    size_t bytes = n * sizeof(float);
    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, bytes));
    CUDA_CHECK(cudaMalloc(&d_y, bytes));
    CUDA_CHECK(cudaMemcpy(d_x, x, bytes, cudaMemcpyHostToDevice));
    int threads = 256, blocks = (n + threads - 1) / threads;
    relu_kernel<<<blocks, threads>>>(d_x, d_y, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(y, d_y, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));
}

__global__ void matmul_kernel(const float* A, const float* B, float* C, int n, int k, int m)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < m) {
        float acc = 0.0f;
        for (int r = 0; r < k; ++r)
            acc += A[row * k + r] * B[r * m + col];
        C[row * m + col] = acc;
    }
}

__global__ void softmax_xent_kernel(const float* Z, const int* y, float* row_loss, float* dZ, int n, int c)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n)
        return;
    const float* z = Z + i * c;
    float largest = z[0];
    for (int j = 1; j < c; ++j)
        largest = fmaxf(largest, z[j]);
    float total = 0.0f;
    for (int j = 0; j < c; ++j)
        total += expf(z[j] - largest);
    float log_total = largest + logf(total);
    row_loss[i] = log_total - z[y[i]];
    for (int j = 0; j < c; ++j)
        dZ[i * c + j] = (expf(z[j] - log_total) - (j == y[i] ? 1.0f : 0.0f)) / n;
}

__global__ void column_sums_kernel(const float* M, float* sums, int n, int m)
{
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j < m) {
        float total = 0.0f;
        for (int i = 0; i < n; ++i)
            total += M[i * m + j];
        sums[j] = total;
    }
}

__global__ void add_bias_kernel(float* Z, const float* b, int n, int m)   // Z[i][j] += b[j]
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n * m)
        Z[i] += b[i % m];
}

#define REDUCE_THREADS 64
// Adds up n numbers with a single block. Each thread first sums a strided slice;
// then the block halves the number of partial sums in shared memory until one is left.
__global__ void sum_kernel(const float* x, float* total, int n)
{
    __shared__ float partial[REDUCE_THREADS];
    int t = threadIdx.x;
    float s = 0.0f;
    for (int i = t; i < n; i += REDUCE_THREADS)
        s += x[i];
    partial[t] = s;
    __syncthreads();
    for (int stride = REDUCE_THREADS / 2; stride > 0; stride /= 2) {
        if (t < stride)
            partial[t] += partial[t + stride];
        __syncthreads();
    }
    if (t == 0)
        *total = partial[0];
}

// C (n x m) = A^T B, where A is k x n and B is k x m: C[i][j] = sum over r of A[r][i] B[r][j].
__global__ void matmul_at_b_kernel(const float* A, const float* B, float* C, int k, int n, int m)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < m) {
        float acc = 0.0f;
        for (int r = 0; r < k; ++r)
            acc += A[r * n + row] * B[r * m + col];
        C[row * m + col] = acc;
    }
}

// C (n x m) = A B^T, where A is n x k and B is m x k: C[i][j] = sum over r of A[i][r] B[j][r].
__global__ void matmul_a_bt_kernel(const float* A, const float* B, float* C, int n, int k, int m)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < m) {
        float acc = 0.0f;
        for (int r = 0; r < k; ++r)
            acc += A[row * k + r] * B[col * k + r];
        C[row * m + col] = acc;
    }
}

// Copies the rows listed in `index` from the training set into the batch buffers.
__global__ void gather_rows_kernel(const float* X, const int* y, const int* index, float* batch_X, int* batch_y,
                                   int batch, int d)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < batch * d) {
        int row = i / d, col = i % d;
        batch_X[i] = X[index[row] * d + col];
        if (col == 0)
            batch_y[row] = y[index[row]];
    }
}

static int blocks(int n) { return (n + 255) / 256; }                       // 256 threads per block
static dim3 grid2d(int cols, int rows) { return dim3((cols + 15) / 16, (rows + 15) / 16); }
static const dim3 block16(16, 16);
"""


def cuda_program(pieces, main):
    """A check program: the prelude, the recap kernels, each (label, source) piece, then main()."""
    return (CHECK4 + mark(RECAP_CU, "recap") + "".join(mark(source, label) for label, source in pieces)
            + mark(main, "the check"))


print("Recap loaded: kernels from notebook 3, the transposed products and the mini-batch gather.")

## 1 · What lives on the device

Every byte that crosses between host and GPU costs time twice: the transfer itself, and the moment the host waits for the GPU to finish before it can copy. A training loop is fast when it rarely does either. The design below keeps all of the training state in device memory for the model's lifetime:

- the **parameters** $W_1, b_1, W_2, b_2$ in one contiguous block, so that one launch can update all of them, followed in the same layout by their **gradients** and Adam's moments $m$ and $v$;
- the **training set**, uploaded once;
- a **workspace** for one mini-batch: the batch itself, $Z_1$, $H$, $Z_2$, the gradients $\bar Z_2$, $\bar H$ and $\bar Z_1$, and the row losses.

Before the loop, `plan_batches` draws the row numbers of every step's mini-batch with the generator (as in notebook 2, its whole state is one 64-bit integer) and sends them all in **one** copy; each step, `gather_rows_kernel` builds its batch on the device from them. The loss of every step stays on the device too, in a history buffer that is copied back once, after the last step. Copying each step's row numbers inside the loop would look harmless, but a synchronous `cudaMemcpy` from ordinary host memory first waits for every kernel already queued, so the host would wait for the GPU on every step. Read the code: it is the scaffold around the function you write in section 4.

In [ ]:
MODEL_CU = r"""
#include <cstdint>
#include <cstdlib>
#include <cstring>

// The whole training state of a d -> h -> c network. Everything marked "device" lives
// in GPU memory for the model's lifetime; only batch indices go in and losses come out.
struct GpuMlp {
    int d, h, c, capacity, rows;
    long step;                                          // Adam's step counter
    uint64_t rng;                                       // splitmix64 state that draws mini-batches
    float lr, beta1, beta2, eps;
    float *params, *grads, *m, *v;                      // device: W1, b1, W2, b2 side by side
    float *W1, *b1, *W2, *b2, *gW1, *gb1, *gW2, *gb2;   // device: pointers into params and grads
    float *X, *bX, *Z1, *H, *Z2, *dZ2, *dH, *dZ1, *row_loss, *history;
    int *y, *by, *plan;                                 // device: labels, batch labels, planned batch rows
    long plan_size;                                     // how many row numbers `plan` can hold
    int history_size;
    float *host_X;                                      // host: a copy of the training set
    int* host_y;
};

static int parameter_count(const GpuMlp* g) { return g->d * g->h + g->h + g->h * g->c + g->c; }

static uint64_t splitmix64(uint64_t* state)
{
    uint64_t z = (*state += 0x9E3779B97F4A7C15ull);
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ull;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBull;
    return z ^ (z >> 31);
}

static void point_into(GpuMlp* g)
{
    g->W1 = g->params;  g->b1 = g->W1 + g->d * g->h;  g->W2 = g->b1 + g->h;  g->b2 = g->W2 + g->h * g->c;
    g->gW1 = g->grads;  g->gb1 = g->gW1 + g->d * g->h;  g->gW2 = g->gb1 + g->h;  g->gb2 = g->gW2 + g->h * g->c;
}

static GpuMlp* allocate(int d, int h, int c, int capacity, float lr, uint64_t seed)
{
    GpuMlp* g = new GpuMlp();
    g->d = d; g->h = h; g->c = c; g->capacity = capacity; g->rows = 0;
    g->step = 0; g->rng = seed;
    g->lr = lr; g->beta1 = 0.9f; g->beta2 = 0.999f; g->eps = 1e-8f;
    size_t p = parameter_count(g) * sizeof(float);
    CUDA_CHECK(cudaMalloc(&g->params, p));
    CUDA_CHECK(cudaMalloc(&g->grads, p));
    CUDA_CHECK(cudaMalloc(&g->m, p));
    CUDA_CHECK(cudaMalloc(&g->v, p));
    CUDA_CHECK(cudaMemset(g->m, 0, p));
    CUDA_CHECK(cudaMemset(g->v, 0, p));
    CUDA_CHECK(cudaMalloc(&g->bX, (size_t)capacity * d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->Z1, (size_t)capacity * h * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->H, (size_t)capacity * h * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->Z2, (size_t)capacity * c * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->dZ2, (size_t)capacity * c * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->dH, (size_t)capacity * h * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->dZ1, (size_t)capacity * h * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->row_loss, (size_t)capacity * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->by, (size_t)capacity * sizeof(int)));
    g->plan = nullptr; g->plan_size = 0;
    g->X = g->bX; g->y = g->by; g->host_X = nullptr; g->host_y = nullptr;   // until data arrives
    g->history = nullptr; g->history_size = 0;
    point_into(g);
    return g;
}

// The forward pass for n rows of x (device memory): logits in g->Z2.
static void forward(GpuMlp* g, const float* x, int n)
{
    matmul_kernel<<<grid2d(g->h, n), block16>>>(x, g->W1, g->Z1, n, g->d, g->h);
    add_bias_kernel<<<blocks(n * g->h), 256>>>(g->Z1, g->b1, n, g->h);
    relu_kernel<<<blocks(n * g->h), 256>>>(g->Z1, g->H, n * g->h);
    matmul_kernel<<<grid2d(g->c, n), block16>>>(g->H, g->W2, g->Z2, n, g->h, g->c);
    add_bias_kernel<<<blocks(n * g->c), 256>>>(g->Z2, g->b2, n, g->c);
}

// The row numbers of `steps` mini-batches of `batch` rows, drawn with replacement by the
// generator (batch 0: every row, in order), sent to the device in ONE copy before training.
// A synchronous cudaMemcpy from ordinary host memory waits for every queued kernel first,
// so copying each step's rows inside the loop would make the host wait on every step.
static const int* plan_batches(GpuMlp* g, int steps, int batch)
{
    int n = batch > 0 ? batch : g->rows;
    long count = batch > 0 ? (long)steps * n : n;
    if (g->plan_size < count) {
        if (g->plan)
            CUDA_CHECK(cudaFree(g->plan));
        CUDA_CHECK(cudaMalloc(&g->plan, count * sizeof(int)));
        g->plan_size = count;
    }
    int* host = new int[count];
    for (long i = 0; i < count; ++i)
        host[i] = batch > 0 ? (int)(splitmix64(&g->rng) % (uint64_t)g->rows) : (int)i;
    CUDA_CHECK(cudaMemcpy(g->plan, host, count * sizeof(int), cudaMemcpyHostToDevice));
    delete[] host;
    return g->plan;
}
"""

### How much memory is that?

Memory is often the first limit a GPU imposes. The function below counts the bytes this design allocates: 16 bytes per parameter (the value, its gradient and two Adam moments, 4 bytes each in `float`), the workspace for a batch, the planned row numbers and losses of one call to `gpu_train` (here 100 steps), and the training set. The second line shows that for a wider network and a larger batch the **activations**, not the weights, dominate, which is why large models need techniques such as activation checkpointing and smaller numeric types.

In [ ]:
def training_memory_bytes(d, h, c, batch, rows, steps_per_call=100):
    """Bytes of device memory a GpuMlp allocates (float32 numbers, int32 labels and row numbers)."""
    params = d * h + h + h * c + c
    weights_state = 4 * params * 4                       # values, gradients and Adam's m and v
    workspace = 4 * batch * (d + 4 * h + 2 * c + 1) + 4 * batch   # floats, then the batch's labels
    plan_and_history = 4 * steps_per_call * (batch + 1)  # row numbers and losses of one gpu_train call
    data = 4 * rows * (d + 1)
    total = weights_state + workspace + plan_and_history + data
    return {"parameters": params, "weights and optimiser": weights_state, "batch workspace": workspace,
            "plan and losses": plan_and_history, "training set": data, "total": total}


for config in [dict(d=2, h=32, c=3, batch=64, rows=600), dict(d=2, h=4096, c=3, batch=8192, rows=600)]:
    sizes = training_memory_bytes(**config)
    print(f"h = {config['h']:>4}, batch = {config['batch']:>4}: {sizes['parameters']:>6} parameters, "
          f"weights and optimiser {sizes['weights and optimiser'] / 1024:7.1f} KiB, "
          f"batch workspace {sizes['batch workspace'] / 1024:8.1f} KiB, total {sizes['total'] / 2**20:6.2f} MiB")

## 2 · The backward pass as kernels

The gradients are the ones you derived in notebook 1, for a batch of $n$ rows:

$$\bar Z_2 = \frac{p - \text{onehot}(y)}{n}, \quad \bar W_2 = H^\top \bar Z_2, \quad \bar b_2 = \textstyle\sum_i \bar Z_{2,i}, \quad \bar H = \bar Z_2 W_2^\top,$$

$$\bar Z_1 = \bar H \odot [Z_1 > 0], \quad \bar W_1 = X^\top \bar Z_1, \quad \bar b_1 = \textstyle\sum_i \bar Z_{1,i}.$$

Every one of them is a kernel you already have: `softmax_xent_kernel` for $\bar Z_2$, the transposed products for $\bar W_2$, $\bar H$ and $\bar W_1$, and `column_sums_kernel` for the biases. One piece is missing: the ReLU's backward rule, an elementwise kernel.

### ✍️ Your turn: `relu_backward_kernel`

Complete `relu_backward_kernel(Z, dH, dZ, n)`: thread `i` (if `i < n`) sets `dZ[i]` to `dH[i]` where `Z[i] > 0.0f`, and to `0.0f` otherwise. `Z` holds the values *before* the ReLU.

*Type it rather than pasting it: typing is how the pattern sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def relu_backward_cu_reference():
    return r"""
__global__ void relu_backward_kernel(const float* Z, const float* dH, float* dZ, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        dZ[i] = Z[i] > 0.0f ? dH[i] : 0.0f;
}
"""

In [ ]:
def relu_backward_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void relu_backward_kernel(const float* Z, const float* dH, float* dZ, int n)
{
    /* ✍️ Your code here */
}
"""

In [ ]:
RELU_BACKWARD_MAIN = r"""
int main()
{
    const int n = 700, guard = 32;
    float Z[n + guard], dH[n + guard], dZ[n + guard];
    fill(Z, n + guard, 0.3f);
    fill(dH, n + guard, 1.9f);
    Z[0] = 0.0f;                                       // exactly zero: the gradient is blocked
    for (int i = 0; i < n + guard; ++i)
        dZ[i] = -99.0f;
    float *d_Z, *d_dH, *d_dZ;
    CUDA_CHECK(cudaMalloc(&d_Z, sizeof Z));
    CUDA_CHECK(cudaMalloc(&d_dH, sizeof dH));
    CUDA_CHECK(cudaMalloc(&d_dZ, sizeof dZ));
    CUDA_CHECK(cudaMemcpy(d_Z, Z, sizeof Z, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_dH, dH, sizeof dH, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_dZ, dZ, sizeof dZ, cudaMemcpyHostToDevice));
    relu_backward_kernel<<<blocks(n), 256>>>(d_Z, d_dH, d_dZ, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(dZ, d_dZ, sizeof dZ, cudaMemcpyDeviceToHost));
    int outside = 0;
    for (int i = n; i < n + guard; ++i)
        outside += dZ[i] != -99.0f;
    show("first 10", dZ, 10);
    show("last 5", dZ + n - 5, 5);
    float total = 0.0f;
    for (int i = 0; i < n; ++i)
        total += dZ[i];
    show("sum", &total, 1);
    std::printf("values written past the end: %d\n", outside);
    return 0;
}
"""


def relu_backward_program(source):
    return cuda_program([("your relu_backward_kernel", source)], RELU_BACKWARD_MAIN)


relu_backward_cu = practice_native("relu_backward_kernel", relu_backward_cu, relu_backward_cu_reference,
                                   relu_backward_program, language="cuda", atol=1e-6, rtol=1e-6)

**Expected behaviour.** ✅, with the gradient passed through where `Z` is positive, blocked elsewhere (including the first entry, where `Z` is exactly 0), and nothing written past the end.

## 3 · Adam as one kernel

Adam treats every parameter entry independently, so it is a single elementwise kernel. Because the parameters, gradients and moments each sit in one contiguous block in the same order, **one launch updates all four weight arrays**. The bias corrections $1 - \beta_1^t$ and $1 - \beta_2^t$ are the same for every entry, so the host computes them once per step, in `double` for accuracy, and passes them in:

$$m_i \leftarrow \beta_1 m_i + (1 - \beta_1) g_i, \qquad v_i \leftarrow \beta_2 v_i + (1 - \beta_2) g_i^2, \qquad \theta_i \leftarrow \theta_i - \eta \, \frac{m_i / (1 - \beta_1^t)}{\sqrt{v_i / (1 - \beta_2^t)} + \varepsilon}.$$

### ✍️ Your turn: `adam_kernel`

Complete `adam_kernel(params, grads, m, v, n, lr, beta1, beta2, eps, correction1, correction2)`: thread `i` (if `i < n`) reads `grads[i]`, updates `m[i]` and `v[i]`, then `params[i]`, with the formulas above. `correction1` and `correction2` are $1 - \beta_1^t$ and $1 - \beta_2^t$. Use `sqrtf`.

*Type it rather than pasting it: typing is how the formulas sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def adam_cu_reference():
    return r"""
__global__ void adam_kernel(float* params, const float* grads, float* m, float* v, int n, float lr,
                            float beta1, float beta2, float eps, float correction1, float correction2)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float g = grads[i];
        m[i] = beta1 * m[i] + (1.0f - beta1) * g;
        v[i] = beta2 * v[i] + (1.0f - beta2) * g * g;
        params[i] -= lr * (m[i] / correction1) / (sqrtf(v[i] / correction2) + eps);
    }
}
"""

In [ ]:
def adam_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void adam_kernel(float* params, const float* grads, float* m, float* v, int n, float lr,
                            float beta1, float beta2, float eps, float correction1, float correction2)
{
    /* ✍️ Your code here */
}
"""

In [ ]:
ADAM_MAIN = r"""
int main()
{
    const int n = 300, guard = 16;
    float params[n + guard], grads[n], m[n], v[n];
    fill(params, n + guard, 0.2f);
    float *d_p, *d_g, *d_m, *d_v;
    CUDA_CHECK(cudaMalloc(&d_p, sizeof params));
    CUDA_CHECK(cudaMalloc(&d_g, sizeof grads));
    CUDA_CHECK(cudaMalloc(&d_m, sizeof m));
    CUDA_CHECK(cudaMalloc(&d_v, sizeof v));
    CUDA_CHECK(cudaMemcpy(d_p, params, sizeof params, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_m, 0, sizeof m));
    CUDA_CHECK(cudaMemset(d_v, 0, sizeof v));
    for (int step = 1; step <= 3; ++step) {
        fill(grads, n, 0.5f * step);
        grads[0] = 0.0f;                               // a zero gradient: no movement at first
        CUDA_CHECK(cudaMemcpy(d_g, grads, sizeof grads, cudaMemcpyHostToDevice));
        float c1 = (float)(1.0 - pow(0.9, step)), c2 = (float)(1.0 - pow(0.999, step));
        adam_kernel<<<blocks(n), 256>>>(d_p, d_g, d_m, d_v, n, 0.01f, 0.9f, 0.999f, 1e-8f, c1, c2);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(params, d_p, sizeof params, cudaMemcpyDeviceToHost));
        char label[32];
        std::snprintf(label, sizeof label, "parameters after step %d", step);
        show(label, params, 8);
    }
    CUDA_CHECK(cudaMemcpy(m, d_m, sizeof m, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(v, d_v, sizeof v, cudaMemcpyDeviceToHost));
    show("first moments", m, 8);
    show("second moments", v, 8);
    float total = 0.0f;
    for (int i = 0; i < n; ++i)
        total += params[i];
    show("sum of the parameters", &total, 1);
    int outside = 0;
    float before[guard];
    fill(params, n + guard, 0.2f);
    for (int i = 0; i < guard; ++i)
        before[i] = params[n + i];
    CUDA_CHECK(cudaMemcpy(params, d_p, sizeof params, cudaMemcpyDeviceToHost));
    for (int i = 0; i < guard; ++i)
        outside += params[n + i] != before[i];
    std::printf("values written past the end: %d\n", outside);
    return 0;
}
"""


def adam_program(source):
    return cuda_program([("your adam_kernel", source)], ADAM_MAIN)


adam_cu = practice_native("adam_kernel", adam_cu, adam_cu_reference, adam_program, language="cuda",
                          atol=1e-6, rtol=1e-5)

**Expected behaviour.** On the first step every entry with a nonzero gradient moves by almost exactly the learning rate, 0.01, against its gradient's sign, and the entry with a zero gradient stays where it was. Nothing beyond the 300 entries changes.

## 4 · One training step without leaving the device

Once `plan_batches` has sent every step's row numbers, a step is a sequence of launches queued on the GPU one after another: `gather_rows_kernel` builds the batch, `forward` runs the five forward launches, and `backward_and_update` runs the rest. Launches on the same stream run in order, so each kernel sees the complete results of the previous one, and the host does not wait for the GPU until the losses are copied at the end.

Two details matter in `backward_and_update`. The step's loss is the **sum** of the row losses, a column sum of an $n \times 1$ matrix, written to the device address `d_loss`; the host divides by $n$ when it finally copies the history. And Adam's step counter lives on the host: increment `g->step`, compute the two corrections in `double`, then launch `adam_kernel` once over all `parameter_count(g)` entries.

### ✍️ Your turn: `backward_and_update`

Complete `backward_and_update(g, n, d_loss)` after the forward pass of `n` batch rows, with the given helpers `blocks(n)` (blocks of 256 threads), `grid2d(cols, rows)` and `block16`. The loss lines are written for you; add, in order:

1. $\bar W_2 = H^\top \bar Z_2$: `matmul_at_b_kernel<<<grid2d(g->c, g->h), block16>>>(g->H, g->dZ2, g->gW2, n, g->h, g->c)`;
2. $\bar b_2$: `column_sums_kernel` of `g->dZ2` ($n \times c$) into `g->gb2`;
3. $\bar H = \bar Z_2 W_2^\top$: `matmul_a_bt_kernel<<<grid2d(g->h, n), block16>>>(g->dZ2, g->W2, g->dH, n, g->c, g->h)`;
4. $\bar Z_1$: your `relu_backward_kernel` over all `n * g->h` entries of `g->Z1` and `g->dH`, into `g->dZ1`;
5. $\bar W_1 = X^\top \bar Z_1$: `matmul_at_b_kernel` with the batch `g->bX` ($n \times d$) and `g->dZ1`, into `g->gW1` ($d \times h$);
6. $\bar b_1$: `column_sums_kernel` of `g->dZ1` into `g->gb1`;
7. the update: `g->step += 1`, the corrections `(float)(1.0 - pow((double)g->beta1, (double)g->step))` and the same with `beta2`, then `adam_kernel` over `g->params`, `g->grads`, `g->m` and `g->v`.

*Type it rather than pasting it: typing is how the order sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def backward_and_update_cu_reference():
    return r"""
void backward_and_update(GpuMlp* g, int n, float* d_loss)
{
    softmax_xent_kernel<<<blocks(n), 256>>>(g->Z2, g->by, g->row_loss, g->dZ2, n, g->c);
    column_sums_kernel<<<1, 32>>>(g->row_loss, d_loss, n, 1);     // the row losses are an n x 1 matrix

    matmul_at_b_kernel<<<grid2d(g->c, g->h), block16>>>(g->H, g->dZ2, g->gW2, n, g->h, g->c);
    column_sums_kernel<<<blocks(g->c), 256>>>(g->dZ2, g->gb2, n, g->c);
    matmul_a_bt_kernel<<<grid2d(g->h, n), block16>>>(g->dZ2, g->W2, g->dH, n, g->c, g->h);
    relu_backward_kernel<<<blocks(n * g->h), 256>>>(g->Z1, g->dH, g->dZ1, n * g->h);
    matmul_at_b_kernel<<<grid2d(g->h, g->d), block16>>>(g->bX, g->dZ1, g->gW1, n, g->d, g->h);
    column_sums_kernel<<<blocks(g->h), 256>>>(g->dZ1, g->gb1, n, g->h);

    g->step += 1;
    float correction1 = (float)(1.0 - pow((double)g->beta1, (double)g->step));
    float correction2 = (float)(1.0 - pow((double)g->beta2, (double)g->step));
    int p = parameter_count(g);
    adam_kernel<<<blocks(p), 256>>>(g->params, g->grads, g->m, g->v, p, g->lr, g->beta1, g->beta2, g->eps,
                                    correction1, correction2);
}
"""

In [ ]:
def backward_and_update_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the function below
    return r"""
void backward_and_update(GpuMlp* g, int n, float* d_loss)
{
    softmax_xent_kernel<<<blocks(n), 256>>>(g->Z2, g->by, g->row_loss, g->dZ2, n, g->c);
    column_sums_kernel<<<1, 32>>>(g->row_loss, d_loss, n, 1);     // the row losses are an n x 1 matrix

    /* ✍️ Your code here: the six launches of the backward pass, then the Adam step */
}
"""

In [ ]:
API_CU = r"""
// The functions Python calls. A GpuMlp travels as an opaque pointer.
extern "C" {

void* gpu_new(int d, int h, int c, int capacity, const float* initial, float lr, uint64_t seed)
{
    clear_errors();
    GpuMlp* g = allocate(d, h, c, capacity, lr, seed);
    CUDA_CHECK(cudaMemcpy(g->params, initial, parameter_count(g) * sizeof(float), cudaMemcpyHostToDevice));
    return g;
}

int gpu_set_data(void* handle, const float* X, const int* y, int rows)
{
    clear_errors();
    GpuMlp* g = (GpuMlp*)handle;
    if (g->host_X) {
        CUDA_CHECK(cudaFree(g->X));
        CUDA_CHECK(cudaFree(g->y));
        delete[] g->host_X;
        delete[] g->host_y;
    }
    g->rows = rows;
    CUDA_CHECK(cudaMalloc(&g->X, (size_t)rows * g->d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&g->y, (size_t)rows * sizeof(int)));
    CUDA_CHECK(cudaMemcpy(g->X, X, (size_t)rows * g->d * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(g->y, y, (size_t)rows * sizeof(int), cudaMemcpyHostToDevice));
    g->host_X = new float[(size_t)rows * g->d];
    g->host_y = new int[rows];
    std::memcpy(g->host_X, X, (size_t)rows * g->d * sizeof(float));
    std::memcpy(g->host_y, y, (size_t)rows * sizeof(int));
    return cuda_status();
}

// `steps` training steps on mini-batches of `batch` rows (0: all rows). The row numbers of
// every step go to the device in one copy before the loop, and the loss of each step stays
// on the device until the end, so the loop does not wait for the GPU; then one copy brings
// back all the losses.
int gpu_train(void* handle, int steps, int batch, float* losses)
{
    clear_errors();
    GpuMlp* g = (GpuMlp*)handle;
    if (g->history_size < steps) {
        if (g->history)
            CUDA_CHECK(cudaFree(g->history));
        CUDA_CHECK(cudaMalloc(&g->history, steps * sizeof(float)));
        g->history_size = steps;
    }
    int n = batch > 0 ? batch : g->rows;
    const int* plan = plan_batches(g, steps, batch);
    for (int s = 0; s < steps; ++s) {
        const int* rows = plan + (batch > 0 ? (size_t)s * n : 0);
        gather_rows_kernel<<<blocks(n * g->d), 256>>>(g->X, g->y, rows, g->bX, g->by, n, g->d);
        forward(g, g->bX, n);
        backward_and_update(g, n, g->history + s);
    }
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(losses, g->history, steps * sizeof(float), cudaMemcpyDeviceToHost));
    for (int s = 0; s < steps; ++s)
        losses[s] /= n;                                 // the device kept sums; report means
    return cuda_status();
}

// The same loop as a naive data pipeline would run it: each batch is gathered on the host and
// uploaded, and each loss is copied back at once, which makes the host wait for the GPU.
int gpu_train_host_batches(void* handle, int steps, int batch, float* losses)
{
    clear_errors();
    GpuMlp* g = (GpuMlp*)handle;
    float* host_batch = new float[(size_t)batch * g->d];
    int* host_labels = new int[batch];
    float* d_loss;
    CUDA_CHECK(cudaMalloc(&d_loss, sizeof(float)));
    for (int s = 0; s < steps; ++s) {
        for (int i = 0; i < batch; ++i) {
            int row = (int)(splitmix64(&g->rng) % (uint64_t)g->rows);
            std::memcpy(host_batch + (size_t)i * g->d, g->host_X + (size_t)row * g->d, g->d * sizeof(float));
            host_labels[i] = g->host_y[row];
        }
        CUDA_CHECK(cudaMemcpy(g->bX, host_batch, (size_t)batch * g->d * sizeof(float), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(g->by, host_labels, batch * sizeof(int), cudaMemcpyHostToDevice));
        forward(g, g->bX, batch);
        backward_and_update(g, batch, d_loss);
        CUDA_CHECK(cudaMemcpy(&losses[s], d_loss, sizeof(float), cudaMemcpyDeviceToHost));
        losses[s] /= batch;
    }
    CUDA_CHECK(cudaFree(d_loss));
    delete[] host_batch;
    delete[] host_labels;
    return cuda_status();
}

// Logits for n rows of host data, computed in chunks of `capacity` rows.
int gpu_logits(void* handle, const float* X, int n, float* out)
{
    clear_errors();
    GpuMlp* g = (GpuMlp*)handle;
    for (int start = 0; start < n; start += g->capacity) {
        int rows = n - start < g->capacity ? n - start : g->capacity;
        CUDA_CHECK(cudaMemcpy(g->bX, X + (size_t)start * g->d, (size_t)rows * g->d * sizeof(float),
                              cudaMemcpyHostToDevice));
        forward(g, g->bX, rows);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(out + (size_t)start * g->c, g->Z2, (size_t)rows * g->c * sizeof(float),
                              cudaMemcpyDeviceToHost));
    }
    return cuda_status();
}

int gpu_parameters(void* handle, float* out)
{
    clear_errors();
    GpuMlp* g = (GpuMlp*)handle;
    CUDA_CHECK(cudaMemcpy(out, g->params, parameter_count(g) * sizeof(float), cudaMemcpyDeviceToHost));
    return cuda_status();
}

// Checkpoint: "FURINAG1", int32 version, int32 d, h, c, capacity, float lr, int64 step,
// uint64 generator state, then the parameters and Adam's two moments (float32, little-endian).
int gpu_save(void* handle, const char* path)
{
    clear_errors();
    GpuMlp* g = (GpuMlp*)handle;
    int p = parameter_count(g);
    float* host = new float[3 * (size_t)p];
    CUDA_CHECK(cudaMemcpy(host, g->params, p * sizeof(float), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(host + p, g->m, p * sizeof(float), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(host + 2 * (size_t)p, g->v, p * sizeof(float), cudaMemcpyDeviceToHost));
    FILE* f = std::fopen(path, "wb");
    int ok = f != nullptr && cuda_status() == 0;
    if (ok) {
        int32_t header[5] = {1, g->d, g->h, g->c, g->capacity};
        int64_t step = g->step;
        ok = std::fwrite("FURINAG1", 1, 8, f) == 8 && std::fwrite(header, sizeof header, 1, f) == 1
             && std::fwrite(&g->lr, sizeof(float), 1, f) == 1 && std::fwrite(&step, sizeof step, 1, f) == 1
             && std::fwrite(&g->rng, sizeof g->rng, 1, f) == 1
             && std::fwrite(host, sizeof(float), 3 * (size_t)p, f) == 3 * (size_t)p;
    }
    if (f)
        ok = (std::fclose(f) == 0) && ok;
    delete[] host;
    return ok ? 0 : 1;
}

void gpu_free(void* handle)
{
    GpuMlp* g = (GpuMlp*)handle;
    if (!g)
        return;
    float* buffers[] = {g->params, g->grads, g->m, g->v, g->bX, g->Z1, g->H, g->Z2, g->dZ2, g->dH, g->dZ1,
                        g->row_loss};
    for (float* buffer : buffers)
        cudaFree(buffer);
    cudaFree(g->by);
    if (g->plan)
        cudaFree(g->plan);
    if (g->history)
        cudaFree(g->history);
    if (g->host_X) {
        cudaFree(g->X);
        cudaFree(g->y);
        delete[] g->host_X;
        delete[] g->host_y;
    }
    delete g;
}

void* gpu_load(const char* path)
{
    clear_errors();
    FILE* f = std::fopen(path, "rb");
    if (!f)
        return nullptr;
    char magic[8];
    int32_t header[5];
    float lr;
    int64_t step;
    uint64_t rng;
    if (std::fread(magic, 1, 8, f) != 8 || std::memcmp(magic, "FURINAG1", 8) != 0
        || std::fread(header, sizeof header, 1, f) != 1 || header[0] != 1
        || std::fread(&lr, sizeof lr, 1, f) != 1 || std::fread(&step, sizeof step, 1, f) != 1
        || std::fread(&rng, sizeof rng, 1, f) != 1) {
        std::fclose(f);
        return nullptr;
    }
    GpuMlp* g = allocate(header[1], header[2], header[3], header[4], lr, rng);
    g->step = step;
    int p = parameter_count(g);
    float* host = new float[3 * (size_t)p];
    bool complete = std::fread(host, sizeof(float), 3 * (size_t)p, f) == 3 * (size_t)p;
    std::fclose(f);
    if (complete) {
        CUDA_CHECK(cudaMemcpy(g->params, host, p * sizeof(float), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(g->m, host + p, p * sizeof(float), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(g->v, host + 2 * (size_t)p, p * sizeof(float), cudaMemcpyHostToDevice));
    }
    delete[] host;
    if (!complete) {                                    // a truncated file is refused
        gpu_free(g);
        return nullptr;
    }
    return g;
}

long gpu_step(void* handle) { return ((GpuMlp*)handle)->step; }
uint64_t gpu_rng(void* handle) { return ((GpuMlp*)handle)->rng; }

}
"""


def training_pieces():
    """Everything the training library needs, with your kernels where they passed their checks."""
    return [("relu_backward_kernel", relu_backward_cu()), ("adam_kernel", adam_cu()), ("the model", MODEL_CU),
            ("backward_and_update", backward_and_update_cu()), ("the interface", API_CU)]


STEP_MAIN = r"""
int main()
{
    const int rows = 40, d = 2, h = 6, c = 3;
    float X[rows * d], params[d * h + h + h * c + c], losses[5], after[d * h + h + h * c + c];
    int y[rows];
    fill(X, rows * d, 0.4f);
    fill(params, d * h + h + h * c + c, 1.1f);
    for (int i = 0; i < rows; ++i)
        y[i] = i % c;
    void* model = gpu_new(d, h, c, 64, params, 0.05f, 7);
    gpu_set_data(model, X, y, rows);
    gpu_train(model, 5, 0, losses);                    // full batches: every row, in order
    show("losses of 5 full-batch steps", losses, 5);
    gpu_train(model, 5, 16, losses);                   // mini-batches of 16 drawn by the generator
    show("losses of 5 mini-batch steps", losses, 5);
    gpu_parameters(model, after);
    show("parameters after 10 steps", after, d * h + h + h * c + c);
    std::printf("Adam steps: %ld\n", gpu_step(model));
    gpu_free(model);
    return 0;
}
"""


def step_program(source):
    return cuda_program([("relu_backward_kernel", relu_backward_cu()), ("adam_kernel", adam_cu()),
                         ("the model", MODEL_CU), ("your backward_and_update", source), ("the interface", API_CU)],
                        STEP_MAIN)


backward_and_update_cu = practice_native("backward_and_update", backward_and_update_cu,
                                         backward_and_update_cu_reference, step_program, language="cuda",
                                         atol=1e-5, rtol=1e-4)

**Expected behaviour.** ✅. The full-batch losses fall step by step, the mini-batch losses vary with the rows drawn, and the counter reads 10. If the first loss matches but later ones do not, the update is wrong: check the order of the launches, the arguments of the transposed products and the bias corrections. The interface code above the check, `API_CU`, holds the functions that Python calls: `gpu_new`, `gpu_set_data`, `gpu_train`, `gpu_logits`, checkpoints and `gpu_free`.

### Check the whole loop in double precision

Matching the reference shows that your function does what the reference does. To check that the reference itself trains the network correctly, the cell below writes the same full-batch Adam training **from scratch in plain Python**, in `double`, and compares 15 steps with the GPU loop on 120 points from the same initial weights. Nothing is shared between the two except the data and the starting point.

In [ ]:
P, I, F = ctypes.c_void_p, ctypes.c_int, ctypes.c_float
pieces = training_pieces()
training_lib = load(LIB4 + mark(RECAP_CU, "recap") + "".join(mark(source, label) for label, source in pieces),
                    "cuda")
for name, argtypes, restype in [
    ("gpu_new", [I, I, I, I, P, F, ctypes.c_uint64], P),
    ("gpu_set_data", [P, P, P, I], I),
    ("gpu_train", [P, I, I, P], I),
    ("gpu_train_host_batches", [P, I, I, P], I),
    ("gpu_logits", [P, P, I, P], I),
    ("gpu_parameters", [P, P], I),
    ("gpu_save", [P, ctypes.c_char_p], I),
    ("gpu_load", [ctypes.c_char_p], P),
    ("gpu_free", [P], None),
    ("gpu_step", [P], ctypes.c_long),
    ("gpu_rng", [P], ctypes.c_uint64),
    ("last_cuda_error", [], ctypes.c_char_p),
]:
    getattr(training_lib, name).argtypes, getattr(training_lib, name).restype = argtypes, restype


def checked(status):
    if status != 0:
        raise RuntimeError(f"CUDA call failed: {training_lib.last_cuda_error().decode()}")


def spirals(per_class, seed, classes=3, noise=0.2, label_noise=0.0):
    gen = random.Random(seed)
    X, y = array("f"), array("i")
    for c in range(classes):
        for i in range(per_class):
            r = i / per_class
            t = 4.0 * c + 4.0 * r + gen.gauss(0.0, noise)
            X.extend((r * math.sin(t), r * math.cos(t)))
            y.append(gen.randrange(classes) if gen.random() < label_noise else c)
    return X, y


def he_init(d, h, c, seed):
    gen = random.Random(seed)
    values = [gen.gauss(0.0, math.sqrt(2.0 / d)) for _ in range(d * h)] + [0.0] * h
    values += [gen.gauss(0.0, math.sqrt(2.0 / h)) for _ in range(h * c)] + [0.0] * c
    return array("f", values)


def reference_training(X, y, params, d, h, c, steps, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    """Full-batch Adam on the same network, in double precision, written from scratch."""
    p, n = list(params), len(y)
    m, v, losses = [0.0] * len(p), [0.0] * len(p), []
    for t in range(1, steps + 1):
        W1, b1 = p[:d * h], p[d * h:d * h + h]
        W2, b2 = p[d * h + h:d * h + h + h * c], p[d * h + h + h * c:]
        gW1, gb1, gW2, gb2, total = [0.0] * (d * h), [0.0] * h, [0.0] * (h * c), [0.0] * c, 0.0
        for i in range(n):
            x = X[i * d:(i + 1) * d]
            z1 = [b1[j] + sum(x[r] * W1[r * h + j] for r in range(d)) for j in range(h)]
            hidden = [max(0.0, z) for z in z1]
            z2 = [b2[k] + sum(hidden[j] * W2[j * c + k] for j in range(h)) for k in range(c)]
            top = max(z2)
            log_total = top + math.log(sum(math.exp(z - top) for z in z2))
            total += log_total - z2[y[i]]
            dz2 = [(math.exp(z - log_total) - (k == y[i])) / n for k, z in enumerate(z2)]
            for j in range(h):
                for k in range(c):
                    gW2[j * c + k] += hidden[j] * dz2[k]
                dh = sum(dz2[k] * W2[j * c + k] for k in range(c))
                dz1 = dh if z1[j] > 0 else 0.0
                for r in range(d):
                    gW1[r * h + j] += x[r] * dz1
                gb1[j] += dz1
            for k in range(c):
                gb2[k] += dz2[k]
        losses.append(total / n)
        g = gW1 + gb1 + gW2 + gb2
        c1, c2 = 1 - beta1 ** t, 1 - beta2 ** t
        for i, gi in enumerate(g):
            m[i] = beta1 * m[i] + (1 - beta1) * gi
            v[i] = beta2 * v[i] + (1 - beta2) * gi * gi
            p[i] -= lr * (m[i] / c1) / (math.sqrt(v[i] / c2) + eps)
    return losses


X_small, y_small = spirals(40, seed=1)
start = he_init(2, 16, 3, seed=2)
model = training_lib.gpu_new(2, 16, 3, 128, address(start), 0.01, 0)
checked(training_lib.gpu_set_data(model, address(X_small), address(y_small), len(y_small)))
gpu_losses = array("f", bytes(4 * 15))
checked(training_lib.gpu_train(model, 15, 0, address(gpu_losses)))
training_lib.gpu_free(model)
ref_losses = reference_training(X_small, y_small, start, 2, 16, 3, 15, lr=0.01)
gap = max(abs(a - b) / b for a, b in zip(gpu_losses, ref_losses))
print("GPU loop:     ", [round(v, 5) for v in gpu_losses[:6]], "...", round(gpu_losses[-1], 5))
print("Python double:", [round(v, 5) for v in ref_losses[:6]], "...", round(ref_losses[-1], 5))
print(f"largest relative gap over 15 steps: {gap:.1e}")
assert gap < 1e-4, "the GPU training loop disagrees with the double-precision reference"

**Expected behaviour.** The two loss sequences agree to about six significant digits over all 15 steps, and the largest relative gap is around $10^{-6}$ or smaller. Single precision rounds every product, so the trajectories drift apart very slowly, but far less than any effect you would report.

## 5 · Train on the GPU, and see where it wins

The cell trains a 2 → 32 → 3 network with Adam on mini-batches of 64 points from 600 spiral points, for 1,500 steps, then scores 900 fresh points. One call runs every step on the device and brings back all 1,500 losses at the end.

In [ ]:
import time


def logits_of(model, X, classes=3):
    out = array("f", bytes(4 * classes * (len(X) // 2)))
    checked(training_lib.gpu_logits(model, address(X), len(X) // 2, address(out)))
    return [list(out[i * classes:(i + 1) * classes]) for i in range(len(X) // 2)]


def accuracy(model, X, y):
    rows = logits_of(model, X)
    return sum(row.index(max(row)) == label for row, label in zip(rows, y)) / len(y)


X_train, y_train = spirals(200, seed=7)
X_fresh, y_fresh = spirals(300, seed=8)
initial = he_init(2, 32, 3, seed=42)
model = training_lib.gpu_new(2, 32, 3, 1024, address(initial), 0.01, 42)
checked(training_lib.gpu_set_data(model, address(X_train), address(y_train), len(y_train)))
losses = array("f", bytes(4 * 1500))
start_time = time.perf_counter()
checked(training_lib.gpu_train(model, 1500, 64, address(losses)))
seconds = time.perf_counter() - start_time
print(f"mean loss of the first 50 steps {sum(losses[:50]) / 50:.3f}, of the last 50 {sum(losses[-50:]) / 50:.3f}; "
      f"{seconds / 1500 * 1e6:.0f} µs per step on {GPU_NAME if CUDA_BACKEND == 'gpu' else 'the CPU emulator'}")
print(f"accuracy {accuracy(model, X_train, y_train):.1%} on the 600 training points, "
      f"{accuracy(model, X_fresh, y_fresh):.1%} on 900 fresh points")
assert accuracy(model, X_fresh, y_fresh) > 0.85, "the GPU loop should learn the spirals"

**Expected behaviour.** The mean loss falls from about 0.76 over the first 50 steps to about 0.03 over the last 50, and the network classifies about 99% of both the training and the fresh points. On a GPU a step takes some tens of microseconds, and almost all of it is **launch overhead**: 15 small kernels per step, each of which the GPU finishes faster than the host can issue the next. A network this small is too little work to keep thousands of threads busy, and a CPU would train it about as fast.

The table below makes that concrete. It times steps for wider networks and larger batches: the GPU's time per step barely moves while the work grows a thousandfold, so its throughput in rows per second climbs. On the emulator only the small configurations run, and the numbers are CPU numbers.

In [ ]:
on_gpu = CUDA_BACKEND == "gpu"
configs = [(32, 64), (256, 1024), (1024, 4096)] if on_gpu else [(32, 64), (128, 256)]
print("Measured on", GPU_NAME if on_gpu else "the CPU emulator: CPU timings, not GPU performance")
for hidden, batch in configs:
    weights = he_init(2, hidden, 3, seed=1)            # kept in a variable while C reads it
    probe = training_lib.gpu_new(2, hidden, 3, batch, address(weights), 0.001, 1)
    checked(training_lib.gpu_set_data(probe, address(X_train), address(y_train), len(y_train)))
    warm, timed = array("f", bytes(4 * 5)), array("f", bytes(4 * 40))
    checked(training_lib.gpu_train(probe, 5, batch, address(warm)))
    start_time = time.perf_counter()
    checked(training_lib.gpu_train(probe, 40, batch, address(timed)))
    per_step = (time.perf_counter() - start_time) / 40
    flops = 6 * batch * (2 * hidden + hidden * 3)       # about 2 for the forward pass and 4 for the backward
    print(f"hidden {hidden:>5}, batch {batch:>5}: {per_step * 1e6:9.0f} µs per step, "
          f"{batch / per_step:12,.0f} rows/s, {flops / per_step / 1e9:7.2f} GFLOP/s")
    training_lib.gpu_free(probe)

**What to notice.** On a GPU, the largest configuration does about a thousand times the work of the smallest in only a few times the time per step. Even at its largest, this network uses a small fraction of a modern GPU's speed: real models reach high utilisation through large matrix products, fused kernels and libraries such as cuBLAS and cuDNN, not through more launches of small ones.

### 🧪 Try it: a training loop that keeps waiting

The library also has `gpu_train_host_batches`, the same training step fed the way a naive data pipeline feeds it: each batch is gathered on the host and uploaded, and each loss is copied back at once. Every step therefore makes the host wait for the GPU, and adds two small transfers. Run the cell, then change `batch` and compare.

In [ ]:
batch = 64  # 🧪 try 1024 on a GPU
for name in ("gpu_train", "gpu_train_host_batches"):
    weights = he_init(2, 32, 3, seed=3)
    probe = training_lib.gpu_new(2, 32, 3, max(batch, 64), address(weights), 0.01, 3)
    checked(training_lib.gpu_set_data(probe, address(X_train), address(y_train), len(y_train)))
    out = array("f", bytes(4 * 300))
    start_time = time.perf_counter()
    checked(getattr(training_lib, name)(probe, 300, batch, address(out)))
    print(f"{name:24}: {(time.perf_counter() - start_time) / 300 * 1e6:7.0f} µs per step, final loss {out[-1]:.3f}")
    training_lib.gpu_free(probe)

**What to notice.** On a GPU, expect the host-fed loop to be clearly slower per step, although the arithmetic is the same: each copy of a single loss forces the host to wait for every queued kernel, so the GPU sits idle between steps while the host prepares the next batch. Keeping data on the device and reading results rarely, or asynchronously, is one of the simplest and largest speed-ups in GPU training. With the emulator the difference mostly disappears, because there is no separate device to wait for.

## 6 · Checkpoints that resume exactly

To checkpoint, the host copies the parameters and Adam's moments from the device, and writes them with the step counter and the generator's state, in a binary format like notebook 2's (with the magic `FURINAG1`). Loading reverses it into a fresh model on the device. Every kernel in this loop adds its numbers in a fixed order, with no atomics, so a resumed run reproduces the original **bit for bit**, on a GPU as well as with the emulator.

The cell saves the trained model, continues the original for 200 steps, loads the checkpoint into a second model, runs the same 200 steps there, and compares the losses with `==`. It then confirms that a damaged file is refused.

In [ ]:
checkpoint = os.path.join(BUILD_DIR, "spirals_gpu.ckpt").encode()
checked(training_lib.gpu_save(model, checkpoint))
continued, resumed = array("f", bytes(4 * 200)), array("f", bytes(4 * 200))
checked(training_lib.gpu_train(model, 200, 64, address(continued)))

restored = training_lib.gpu_load(checkpoint)
assert restored, "the checkpoint could not be loaded"
checked(training_lib.gpu_set_data(restored, address(X_train), address(y_train), len(y_train)))
print("restored step and generator state:", training_lib.gpu_step(restored), training_lib.gpu_rng(restored))
checked(training_lib.gpu_train(restored, 200, 64, address(resumed)))
identical = list(continued) == list(resumed)
print(f"200 losses after the checkpoint: identical = {identical}")

damaged = os.path.join(BUILD_DIR, "damaged_gpu.ckpt")
with open(checkpoint, "rb") as source, open(damaged, "wb") as target:
    target.write(source.read()[:100])                  # a file cut short
print("loading a truncated checkpoint returns:", training_lib.gpu_load(damaged.encode()))
training_lib.gpu_free(restored)
training_lib.gpu_free(model)
assert identical

**Expected behaviour.** The restored model reports step 1,500 and the saved generator state, the 200 losses are identical, and the truncated file is refused with `None`. The training set is not part of the checkpoint, so it is uploaded again after loading. A checkpoint for inference alone would need only the parameters and the sizes; this one also holds Adam's state, so that training can continue.

## 7 · Select on validation, test once, verify the artefact

The evaluation protocol is the one from *Base Model Training*, now with the model on the GPU:

- **select** on validation data only: train in chunks, measure the validation loss after each chunk, and keep the best checkpoint as a **file**. Writing it copies the weights off the device, so later updates cannot change it, which is exactly what a pointer to the live weights would fail to do;
- **test once**: the test rows sit in an object that hands them out a single time, after every decision is made;
- **look beyond one number**: a confusion matrix, per-class recall, and the most confident mistakes;
- **verify the artefact**: reload the selected checkpoint into a fresh model and compare its logits on fixed inputs with the logits computed before saving. Same weights and same kernels give the same numbers, so the difference must be exactly 0.

The helpers below implement the selection loop and the sealed test set. You write the confusion matrix.

In [ ]:
class SealedTestSet:
    """Test rows that can be opened exactly once."""

    def __init__(self, X, y):
        self._X, self._y, self.times_opened = X, y, 0

    def open(self):
        if self.times_opened:
            raise RuntimeError("This test set has already been used once.")
        self.times_opened += 1
        return self._X, self._y


def mean_cross_entropy(rows, y):
    total = 0.0
    for row, label in zip(rows, y):
        top = max(row)
        total += top + math.log(sum(math.exp(v - top) for v in row)) - row[label]
    return total / len(y)


def train_and_select(config, seed, X_tr, y_tr, X_val, y_val, steps, chunk=100, tag="run"):
    """Train on the GPU in chunks; keep the checkpoint with the lowest validation loss as a file."""
    h, lr = config["hidden"], config["learning_rate"]
    weights = he_init(2, h, 3, seed=1000 + seed)
    run = training_lib.gpu_new(2, h, 3, 1024, address(weights), lr, seed)
    checked(training_lib.gpu_set_data(run, address(X_tr), address(y_tr), len(y_tr)))
    best = {"val_loss": math.inf, "step": 0, "path": os.path.join(BUILD_DIR, f"{tag}_seed{seed}_best.ckpt")}
    losses = array("f", bytes(4 * chunk))
    for _ in range(steps // chunk):
        checked(training_lib.gpu_train(run, chunk, 64, address(losses)))
        val_loss = mean_cross_entropy(logits_of(run, X_val), y_val)
        if val_loss < best["val_loss"]:
            best.update(val_loss=val_loss, step=training_lib.gpu_step(run))
            checked(training_lib.gpu_save(run, best["path"].encode()))
    training_lib.gpu_free(run)
    return best


print("SealedTestSet, mean_cross_entropy and train_and_select are defined.")

### ✍️ Your turn: `confusion_matrix`

Write `confusion_matrix(y_true, y_pred, classes)` in Python:

- **inputs:** two sequences of class indices of the same length, and the number of classes;
- **output:** a list of `classes` lists of `classes` integers, where entry `[t][p]` counts the rows whose true class is `t` and whose predicted class is `p`. The diagonal holds the correct predictions.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def confusion_matrix_reference(y_true, y_pred, classes):
    matrix = [[0] * classes for _ in range(classes)]
    for t, p in zip(y_true, y_pred):
        matrix[t][p] += 1
    return matrix

In [ ]:
def confusion_matrix(y_true, y_pred, classes):
    """counts[t][p]: rows of true class t predicted as class p."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
confusion_matrix = practice("confusion_matrix", confusion_matrix, confusion_matrix_reference, cases=[
    ([0, 1, 1, 0, 1], [0, 1, 0, 1, 1], 2),
    ([2, 0, 2, 1], [2, 0, 1, 1], 3),
    ([], [], 2),                     # edge case: no rows at all
])

**Expected behaviour.** For the five-row example the matrix is `[[1, 1], [1, 2]]`: one row of class 0 predicted correctly and one predicted as class 1, and two of the three class-1 rows predicted correctly. With no rows, every count is 0.

## 8 · Capstone: a reproducible base model on the GPU

The capstone is a small but complete experiment that someone else could reproduce and question. The scaffold runs from start to finish with default choices on a **new** task, so that it has its own unused test set: three spiral arms with 5% of the labels replaced at random, split by row into training, validation and test partitions. The rows are independent draws, so a split by row does not leak; with grouped data, as in *Base Model Training*, you would split by source instead.

Work through it in order:

1. **Write the protocol first.** Edit the `protocol` cell *before* you look at any result: the split, the baseline, the validation metric, the budget, the selection rule, the single controlled change and the seeds.
2. **Run the comparison on validation data only.** Both configurations train with three seeds each, on whichever backend this notebook uses; nothing touches the test rows.
3. **Decide, then test once.** The configuration with the lower mean validation loss wins, and its seed-0 checkpoint becomes the artefact. Only then is the sealed test set opened.
4. **Verify the artefact** by reloading it and comparing logits on fixed inputs.
5. **Report**, including the backend and the toolchain, in your own words.

In [ ]:
protocol = {
    "task": "three spiral arms, 400 points per arm; generated; 5% of the labels replaced at random",
    "split": "by row (independent draws): 60% training, 20% validation, 20% test; split seed 31",
    "baseline": "always predict the most common training class",
    "primary_metric": "validation cross-entropy of each run's selected checkpoint, averaged over the seeds",
    "budget": "1,500 Adam steps of 64 rows per run; validation every 100 steps",
    "selection": "within a run, the lowest validation loss; between configurations, the lower mean over seeds",
    "base_config": {"hidden": 32, "learning_rate": 0.01},
    "controlled_change": {"hidden": 8},   # 🧪 one change only, such as {"learning_rate": 0.003}
    "seeds": [0, 1, 2],
    "artefact": "the seed-0 run of the winning configuration",
}
for key, value in protocol.items():
    print(f"{key:>17}: {value}")

In [ ]:
X_all, y_all = spirals(400, seed=30, label_noise=0.05)
order = list(range(len(y_all)))
random.Random(31).shuffle(order)
cut1, cut2 = int(0.6 * len(order)), int(0.8 * len(order))


def subset(rows):
    X, y = array("f"), array("i")
    for i in rows:
        X.extend(X_all[2 * i:2 * i + 2])
        y.append(y_all[i])
    return X, y


X_tr, y_tr = subset(order[:cut1])
X_val, y_val = subset(order[cut1:cut2])
sealed_test = SealedTestSet(*subset(order[cut2:]))
majority = max(range(3), key=list(y_tr).count)
print(f"{len(y_tr)} training, {len(y_val)} validation and {len(order) - cut2} sealed test rows; "
      f"majority baseline on validation {sum(v == majority for v in y_val) / len(y_val):.1%}")

configurations = {"base": protocol["base_config"],
                  "changed": {**protocol["base_config"], **protocol["controlled_change"]}}
capstone = {}
for name, config in configurations.items():
    runs = [train_and_select(config, seed, X_tr, y_tr, X_val, y_val, steps=1500, tag=name)
            for seed in protocol["seeds"]]
    losses = [run["val_loss"] for run in runs]
    mean = sum(losses) / len(losses)
    spread = math.sqrt(sum((v - mean) ** 2 for v in losses) / len(losses))
    capstone[name] = {"config": config, "runs": runs, "mean": mean, "spread": spread}
    print(f"{name:8} {config}: validation loss {mean:.4f} ± {spread:.4f} "
          f"(selected steps {[run['step'] for run in runs]})")

In [ ]:
def toolchain():
    """What produced these numbers: Python, the compiler and the CUDA backend."""
    first_line = lambda command: subprocess.run(command, capture_output=True, text=True).stdout.strip().splitlines()[0]
    found = {"python": sys.version.split()[0], "c++ compiler": first_line([CXX, "--version"])}
    if CUDA_BACKEND == "gpu":
        found["gpu"] = GPU_NAME
        found["nvcc"] = next(line for line in subprocess.run([NVCC, "--version"], capture_output=True,
                                                              text=True).stdout.splitlines() if "release" in line)
    else:
        found["cuda backend"] = "CPU emulator (no NVIDIA GPU)"
    return found


winner = min(capstone, key=lambda name: capstone[name]["mean"])
gap = abs(capstone["base"]["mean"] - capstone["changed"]["mean"])
spread = max(capstone["base"]["spread"], capstone["changed"]["spread"])
artefact_path = capstone[winner]["runs"][0]["path"]           # decided in the protocol: the seed-0 run
artefact = training_lib.gpu_load(artefact_path.encode())

X_test, y_test = sealed_test.open()                          # once, after the decision
test_rows = logits_of(artefact, X_test)
predictions = [row.index(max(row)) for row in test_rows]
matrix = confusion_matrix(list(y_test), predictions, 3)
recall = [matrix[k][k] / max(sum(matrix[k]), 1) for k in range(3)]
report = {
    "protocol": protocol,
    "validation": {name: {"mean_loss": round(c["mean"], 4), "spread": round(c["spread"], 4)}
                   for name, c in capstone.items()},
    "decision": winner,
    "difference_vs_seed_spread": [round(gap, 4), round(spread, 4)],
    "test": {"rows": len(y_test), "accuracy": round(sum(p == t for p, t in zip(predictions, y_test)) / len(y_test), 4),
             "majority_baseline": round(sum(t == majority for t in y_test) / len(y_test), 4),
             "confusion_matrix": matrix, "recall": [round(r, 3) for r in recall]},
    "toolchain": toolchain(),
}
for key in ("validation", "decision", "difference_vs_seed_spread", "test", "toolchain"):
    print(f"{key}: {report[key]}")
assert sealed_test.times_opened == 1 and report["test"]["accuracy"] > report["test"]["majority_baseline"]

In [ ]:
fixed_inputs = X_val[:40]                                     # 20 fixed validation rows
logits_saved = logits_of(artefact, fixed_inputs)
rebuilt = training_lib.gpu_load(artefact_path.encode())       # a fresh model from the file alone
logits_rebuilt = logits_of(rebuilt, fixed_inputs)
largest_gap = max(abs(a - b) for row_a, row_b in zip(logits_saved, logits_rebuilt) for a, b in zip(row_a, row_b))
report["reload"] = {"verified": largest_gap == 0.0, "largest_logit_gap": largest_gap}
print("reload:", report["reload"])


def most_confident_mistakes(rows, y, k):
    """Test rows predicted wrongly, the most confident first: (row, true class, predicted class, probability)."""
    found = []
    for i, (row, label) in enumerate(zip(rows, y)):
        top = max(row)
        probs = [math.exp(v - top) for v in row]
        predicted = probs.index(max(probs))
        if predicted != label:
            found.append((i, label, predicted, round(max(probs) / sum(probs), 3)))
    return sorted(found, key=lambda item: -item[3])[:k]


print("three most confident test mistakes:", most_confident_mistakes(test_rows, y_test, 3))
training_lib.gpu_free(rebuilt)
training_lib.gpu_free(artefact)
assert report["reload"]["verified"], "the reloaded artefact must reproduce its logits exactly"

**What to notice in the default run.** The wider network usually reaches a lower validation loss than the narrow one, by more than the spread across seeds, but check what *your* run printed before you draw that conclusion. The test accuracy describes only the chosen artefact, on this generated task: with 5% of the labels replaced at random, no model can be right on every test row, and the most confident mistakes are often exactly those replaced labels. The reloaded model reproduces its logits with a gap of exactly 0. Change `controlled_change` (one change only), run the capstone cells again from the protocol cell down, and compare.

### Your report

Answer each prompt in two to four sentences, in a markdown cell of your own, using the numbers that *your* run printed:

1. **Protocol.** What was decided before any result was seen, and why is a split by row acceptable here?
2. **Result.** What did the controlled change do to the mean validation loss, compared with the spread across seeds? Was it an improvement, a failure, or no detectable difference?
3. **Test.** What was the test accuracy of the artefact, against the majority baseline and with the number of test rows? Why was the test set opened only after the decision?
4. **Three mistakes.** Look at the three most confident mistakes: their rows in `X_test` and their labels. Are they replaced labels, points near another arm, or something else?
5. **Backend and limitations.** Which backend produced the numbers (the `toolchain` entry)? What does this experiment *not* show, given the generated data, its size, and single-precision arithmetic?
6. **Reproduction.** Restart the kernel and run every cell again: is the report identical? On a GPU, would it still be identical without the fixed-order sums of this loop?

## Exercises

### Exercise 1: `adamw_kernel`, decoupled weight decay

AdamW adds weight decay *outside* the adaptive step: after the Adam update, each parameter also shrinks by $\eta \lambda \theta_i$, where $\lambda$ is the decay rate. Write `adamw_kernel`, with the same arguments as `adam_kernel` plus a final `float weight_decay`, that performs the Adam update and then `params[i] -= lr * weight_decay * params[i]`, using the parameter value from *before* the update for the decay, as the AdamW paper does.

*Type it rather than pasting it: typing is how the formula sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def adamw_cu_reference():
    return r"""
__global__ void adamw_kernel(float* params, const float* grads, float* m, float* v, int n, float lr,
                             float beta1, float beta2, float eps, float correction1, float correction2,
                             float weight_decay)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float g = grads[i], before = params[i];
        m[i] = beta1 * m[i] + (1.0f - beta1) * g;
        v[i] = beta2 * v[i] + (1.0f - beta2) * g * g;
        params[i] = before - lr * (m[i] / correction1) / (sqrtf(v[i] / correction2) + eps)
                    - lr * weight_decay * before;
    }
}
"""

In [ ]:
def adamw_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void adamw_kernel(float* params, const float* grads, float* m, float* v, int n, float lr,
                             float beta1, float beta2, float eps, float correction1, float correction2,
                             float weight_decay)
{
    /* ✍️ Your code here */
}
"""

In [ ]:
ADAMW_MAIN = r"""
int main()
{
    const int n = 50;
    float params[n], grads[n];
    fill(params, n, 0.6f);
    fill(grads, n, 1.3f);
    float *d_p, *d_g, *d_m, *d_v;
    CUDA_CHECK(cudaMalloc(&d_p, sizeof params));
    CUDA_CHECK(cudaMalloc(&d_g, sizeof grads));
    CUDA_CHECK(cudaMalloc(&d_m, sizeof params));
    CUDA_CHECK(cudaMalloc(&d_v, sizeof params));
    CUDA_CHECK(cudaMemcpy(d_p, params, sizeof params, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_g, grads, sizeof grads, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_m, 0, sizeof params));
    CUDA_CHECK(cudaMemset(d_v, 0, sizeof params));
    for (int step = 1; step <= 2; ++step) {
        float c1 = (float)(1.0 - pow(0.9, step)), c2 = (float)(1.0 - pow(0.999, step));
        adamw_kernel<<<blocks(n), 256>>>(d_p, d_g, d_m, d_v, n, 0.01f, 0.9f, 0.999f, 1e-8f, c1, c2, 0.1f);
    }
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(params, d_p, sizeof params, cudaMemcpyDeviceToHost));
    show("parameters after 2 AdamW steps", params, n);
    return 0;
}
"""


def adamw_program(source):
    return cuda_program([("your adamw_kernel", source)], ADAMW_MAIN)


adamw_cu = practice_native("adamw_kernel", adamw_cu, adamw_cu_reference, adamw_program, language="cuda",
                           atol=1e-6, rtol=1e-5)

### Exercise 2: an open question

The GPU loop in this notebook is deterministic because every sum has a fixed order. Suppose you replace `column_sums_kernel` in the backward pass with the atomic version from notebook 3, which is faster for very tall matrices. What would happen to the exact-resume test of section 6, and to the capstone's reproduction prompt? How would you decide whether the speed is worth it, and what would you write in the report?

## Common errors and debugging

- **The loss stays flat, or turns to `nan` after a few steps.** The update or a gradient is wrong. Compare a few steps with the double-precision reference of section 4 before training longer; a gradient that is too large by a factor of $n$ usually means a missing division by the batch size.
- **Results that differ between two identical runs on a GPU.** Something adds in a varying order, typically `atomicAdd`, or a kernel has a race. The loop in this notebook avoids both, which is what makes exact resumption possible.
- **A training loop that is slow on a GPU although its kernels are fast.** The host waits every step: a copy of the loss, an upload of a batch, or a `cudaDeviceSynchronize`. Keep data on the device and read results once, as `gpu_train` does.
- **`CUDA error: out of memory`.** The activations of a large batch fill the device. Estimate them first with `training_memory_bytes`, then reduce the batch or the width.
- **A reloaded model whose logits differ from the saved model's.** The checkpoint missed a piece of state, the sizes do not match the parameters, or a different kernel computes the forward pass. With the same weights and kernels, the gap is exactly 0.
- **`RuntimeError: This test set has already been used once`.** A cell that opens the sealed test set ran twice. That is the point of the guard; run the notebook again from the top for a clean start.
- **Colab reports the CPU emulator although you chose a GPU.** The runtime changed after the setup cells ran, or the GPU quota is used up. Run the notebook again from the top and read the backend line.

## Summary

- A fast GPU training loop keeps its data, weights, gradients, optimiser state and intermediate results in device memory, sends the batch row numbers in one copy before the loop, and reads the losses back once. Its memory is 16 bytes per parameter plus the activations of a batch, which dominate for wide layers and large batches.
- The backward pass is a sequence of kernels you already know: transposed products, column sums, an elementwise ReLU rule, and one Adam launch over all contiguous parameters. An independent double-precision implementation confirms the whole loop.
- GPUs win when each step has enough parallel work; small models are bound by launch overhead, and a loop that waits for the GPU every step wastes much of its speed.
- Fixed-order sums make GPU training deterministic, so a checkpoint with the parameters, Adam's moments, the step and the generator's state resumes it bit for bit.
- The evaluation protocol does not change on a GPU: select on validation data, test once, verify a reloaded artefact exactly, and report the backend with the results.

## Next

You have built a training stack from the bottom: loops over raw memory in C, an autograd engine with RAII in C++, CUDA kernels with shared memory, and a training loop that never leaves the GPU, verified at every step. The *Intermediate Model Training* course improves a working model through controlled experiments, and its ideas, from learning-rate schedules to mixed precision, map directly onto the kernels you wrote here.